In [35]:
import numpy as np
import pandas as pd
from io import StringIO
from unlzw3 import unlzw
from scipy.io import loadmat
from scipy.sparse import lil_matrix, coo_matrix
from sklearn.model_selection import train_test_split

In [3]:
RANDOM_STATE = 42
TEST_SIZE = 0.3

# Parkinson's Disease Classification

In [25]:
data = pd.read_csv("../Datasets/Parkinsons_Disease_Classification/pd_speech_features/pd_speech_features.csv", skiprows=1)
data

,id,gender,PPE,DFA,RPDE,numPulses,numPeriodsPulses,meanPeriodPulses,stdDevPeriodPulses,locPctJitter,...,tqwt_kurtosisValue_dec_28,tqwt_kurtosisValue_dec_29,tqwt_kurtosisValue_dec_30,tqwt_kurtosisValue_dec_31,tqwt_kurtosisValue_dec_32,tqwt_kurtosisValue_dec_33,tqwt_kurtosisValue_dec_34,tqwt_kurtosisValue_dec_35,tqwt_kurtosisValue_dec_36,class
0,0,1,0.85247,0.71826,0.57227,240,239,0.008064,0.000087,0.00218,...,1.5620,2.6445,3.8686,4.2105,5.1221,4.4625,2.6202,3.0004,18.9405,1
1,0,1,0.76686,0.69481,0.53966,234,233,0.008258,0.000073,0.00195,...,1.5589,3.6107,23.5155,14.1962,11.0261,9.5082,6.5245,6.3431,45.1780,1
2,0,1,0.85083,0.67604,0.58982,232,231,0.008340,0.000060,0.00176,...,1.5643,2.3308,9.4959,10.7458,11.0177,4.8066,2.9199,3.1495,4.7666,1
3,1,0,0.41121,0.79672,0.59257,178,177,0.010858,0.000183,0.00419,...,3.7805,3.5664,5.2558,14.0403,4.2235,4.6857,4.8460,6.2650,4.0603,1
4,1,0,0.32790,0.79782,0.53028,236,235,0.008162,0.002669,0.00535,...,6.1727,5.8416,6.0805,5.7621,7.7817,11.6891,8.2103,5.0559,6.1164,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
751,250,0,0.80903,0.56355,0.28385,417,416,0.004627,0.000052,0.00064,...,3.0706,3.0190,3.1212,2.4921,3.5844,3.5400,3.3805,3.2003,6.8671,0
752,250,0,0.16084,0.56499,0.59194,415,413,0.004550,0.000220,0.00143,...,1.9704,1.7451,1.8277,2.4976,5.2981,4.2616,6.3042,10.9058,28.4170,0
753,251,0,0.88389,0.72335,0.46815,381,380,0.005069,0.000103,0.00076,...,51.5607,44.4641,26.1586,6.3076,2.8601,2.5361,3.5377,3.3545,5.0424,0
754,251,0,0.83782,0.74890,0.49823,340,339,0.005679,0.000055,0.00092,...,19.1607,12.8312,8.9434,2.2044,1.9496,1.9664,2.6801,2.8332,3.7131,0


In [27]:
data.groupby("id")["class"].nunique().nunique()

1

In [28]:
data.rename(columns={"class": "label"}, inplace=True)

In [29]:
groups = data.groupby("id")["label"].first().reset_index()

# Stratified split by label, not by individual rows
train_ids, test_ids = train_test_split(
    groups["id"],
    test_size=TEST_SIZE,
    stratify=groups["label"],
    random_state=RANDOM_STATE
)

# Select all rows that belong to those molecules
train_df = data[data["id"].isin(train_ids)].copy()
test_df = data[data["id"].isin(test_ids)].copy()

print(f"Train ids: {len(train_ids)}, Test ids: {len(test_ids)}")
print(f"Train rows: {len(train_df)}, Test rows: {len(test_df)}")

Train ids: 176, Test ids: 76
Train rows: 528, Test rows: 228


In [30]:
set(train_df["id"]).intersection(set(test_df["id"])), set(train_df["id"]).intersection(set(test_df["id"]))

(set(), set())

In [31]:
train_df["subset"] = "train"
test_df["subset"] = "test"
data = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)
data.drop(columns=["id"], inplace=True)
data.columns = [col if col in ["label", "subset"] else i for i, col in enumerate(data.columns)]
data

,0,1,2,3,4,5,6,7,8,9,...,745,746,747,748,749,750,751,752,label,subset
0,1,0.85247,0.71826,0.57227,240,239,0.008064,0.000087,0.00218,0.000018,...,2.6445,3.8686,4.2105,5.1221,4.4625,2.6202,3.0004,18.9405,1,train
1,1,0.76686,0.69481,0.53966,234,233,0.008258,0.000073,0.00195,0.000016,...,3.6107,23.5155,14.1962,11.0261,9.5082,6.5245,6.3431,45.1780,1,train
2,1,0.85083,0.67604,0.58982,232,231,0.008340,0.000060,0.00176,0.000015,...,2.3308,9.4959,10.7458,11.0177,4.8066,2.9199,3.1495,4.7666,1,train
3,1,0.76095,0.62145,0.54543,322,321,0.005991,0.000107,0.00222,0.000013,...,75.3156,32.0478,7.7060,3.1060,4.6206,12.8353,13.8300,7.7693,1,train
4,1,0.83671,0.62079,0.51179,318,317,0.006074,0.000136,0.00282,0.000017,...,11.8909,7.2891,4.3682,3.6443,5.9610,11.7552,18.0927,5.0448,1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
751,0,0.80903,0.56355,0.28385,417,416,0.004627,0.000052,0.00064,0.000003,...,3.0190,3.1212,2.4921,3.5844,3.5400,3.3805,3.2003,6.8671,0,test
752,0,0.16084,0.56499,0.59194,415,413,0.004550,0.000220,0.00143,0.000006,...,1.7451,1.8277,2.4976,5.2981,4.2616,6.3042,10.9058,28.4170,0,test
753,0,0.88389,0.72335,0.46815,381,380,0.005069,0.000103,0.00076,0.000004,...,44.4641,26.1586,6.3076,2.8601,2.5361,3.5377,3.3545,5.0424,0,test
754,0,0.83782,0.74890,0.49823,340,339,0.005679,0.000055,0.00092,0.000005,...,12.8312,8.9434,2.2044,1.9496,1.9664,2.6801,2.8332,3.7131,0,test


In [32]:
data["subset"].value_counts(normalize=True)

subset
train    0.698413
test     0.301587
Name: proportion, dtype: float64

In [33]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('int64'), dtype('float64')], dtype=object),
 label
 1    0.746032
 0    0.253968
 Name: proportion, dtype: float64,
 subset
 train    0.698413
 test     0.301587
 Name: proportion, dtype: float64,
 subset  label
 test    1        0.750000
         0        0.250000
 train   1        0.744318
         0        0.255682
 Name: proportion, dtype: float64,
 np.int64(0))

In [34]:
data.to_parquet("../Benchmark/parkinson_disease.parquet", index=False)

# LSVT Voice Rehabilitation

In [20]:
input_data = pd.read_excel("../Datasets/lsvt+voice+rehabilitation/LSVT_voice_rehabilitation.xlsx", sheet_name="Data")
input_data.head(3)

,Jitter->F0_abs_dif,Jitter->F0_dif_percent,Jitter->F0_PQ5_classical_Schoentgen,Jitter->F0_PQ5_classical_Baken,Jitter->F0_PQ5_generalised_Schoentgen,Jitter->F0_abs0th_perturb,Jitter->F0_CV,Jitter->F0_TKEO_mean,Jitter->F0_TKEO_std,Jitter->F0_TKEO_prc5,...,det_TKEO_std4_1_coef,det_TKEO_std4_2_coef,det_TKEO_std4_3_coef,det_TKEO_std4_4_coef,det_TKEO_std4_5_coef,det_TKEO_std4_6_coef,det_TKEO_std4_7_coef,det_TKEO_std4_8_coef,det_TKEO_std4_9_coef,det_TKEO_std4_10_coef
0,0.088112,0.041697,0.000480,-3.723304e-06,0.000422,2.458381,6.332164e-07,47.021079,1366.430390,-7.103323,...,2.527583,7.088978,19.753255,54.335046,145.528630,375.097397,921.296579,2137.079844,4697.131077,9931.208257
1,0.161798,0.057364,0.000677,5.466365e-06,0.000206,2.592066,7.228518e-07,93.557936,2582.922776,-23.284761,...,2.841881,7.977363,22.203504,60.993338,163.560972,421.010306,1036.092589,2404.072562,5284.082128,11165.095662
2,0.554508,0.642913,0.007576,-7.443871e-07,0.006488,12.691326,6.946246e-04,52.988422,466.682635,-45.308680,...,1.806103,5.078616,14.135923,38.641654,103.466808,264.654626,649.657090,1507.384591,3315.804236,6974.600636


In [21]:
output_data = pd.read_excel("../Datasets/lsvt+voice+rehabilitation/LSVT_voice_rehabilitation.xlsx", sheet_name="Binary response")
output_data.head(3)

,"Binary class 1=acceptable, 2=unacceptable"
0,1
1,2
2,2


In [23]:
input_data.columns = np.arange(input_data.shape[1])
input_data.head(3)

,0,1,2,3,4,5,6,7,8,9,...,300,301,302,303,304,305,306,307,308,309
0,0.088112,0.041697,0.000480,-3.723304e-06,0.000422,2.458381,6.332164e-07,47.021079,1366.430390,-7.103323,...,2.527583,7.088978,19.753255,54.335046,145.528630,375.097397,921.296579,2137.079844,4697.131077,9931.208257
1,0.161798,0.057364,0.000677,5.466365e-06,0.000206,2.592066,7.228518e-07,93.557936,2582.922776,-23.284761,...,2.841881,7.977363,22.203504,60.993338,163.560972,421.010306,1036.092589,2404.072562,5284.082128,11165.095662
2,0.554508,0.642913,0.007576,-7.443871e-07,0.006488,12.691326,6.946246e-04,52.988422,466.682635,-45.308680,...,1.806103,5.078616,14.135923,38.641654,103.466808,264.654626,649.657090,1507.384591,3315.804236,6974.600636


In [24]:
output_data.columns = ["label"]
# Binary class 1=acceptable, 2=unacceptable
output_data.loc[output_data["label"] == 2, "label"] = 0

In [28]:
data = pd.concat([input_data, output_data], axis=1)
data

,0,1,2,3,4,5,6,7,8,9,...,301,302,303,304,305,306,307,308,309,label
0,0.088112,0.041697,0.000480,-3.723304e-06,0.000422,2.458381,6.332164e-07,47.021079,1366.430390,-7.103323,...,7.088978,19.753255,54.335046,145.528630,375.097397,921.296579,2137.079844,4697.131077,9931.208257,1
1,0.161798,0.057364,0.000677,5.466365e-06,0.000206,2.592066,7.228518e-07,93.557936,2582.922776,-23.284761,...,7.977363,22.203504,60.993338,163.560972,421.010306,1036.092589,2404.072562,5284.082128,11165.095662,0
2,0.554508,0.642913,0.007576,-7.443871e-07,0.006488,12.691326,6.946246e-04,52.988422,466.682635,-45.308680,...,5.078616,14.135923,38.641654,103.466808,264.654626,649.657090,1507.384591,3315.804236,6974.600636,0
3,0.031089,0.027108,0.000314,-2.214722e-07,0.000216,0.754288,1.868647e-07,13.982754,417.217249,-1.207741,...,5.610448,15.626164,42.943275,115.014975,296.320795,728.284936,1689.586636,3713.818933,7851.139360,1
4,0.076177,0.039071,0.000302,2.732106e-05,0.001102,1.270034,4.918186e-05,56.373996,1608.317410,-3.491990,...,6.902199,19.117609,52.715873,141.113865,363.511021,893.246151,2071.625622,4554.204815,9623.566242,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,0.116214,0.070546,0.000837,2.765074e-06,0.000333,1.890006,9.844257e-07,34.083311,896.672947,-10.446397,...,6.527285,18.210931,50.085484,134.123291,345.396264,850.942761,1973.383824,4336.099395,9158.984652,0
122,0.700258,0.334397,0.003959,8.297261e-06,0.001516,4.557797,1.581899e-05,104.648435,1583.166169,-97.281717,...,7.160142,20.081389,55.178858,147.584708,379.897760,935.982559,2166.960428,4769.956102,10067.750435,0
123,0.072635,0.050743,0.000597,-5.277518e-06,0.000434,6.984651,4.993260e-07,21.859427,625.288493,-4.116001,...,6.141876,17.095325,46.893015,125.687344,323.728298,795.715774,1845.609006,4056.256338,8583.121863,1
124,0.111362,0.054237,0.000646,-1.546671e-06,0.000277,1.935398,4.398078e-07,47.870508,1367.843467,-9.373059,...,7.125849,19.849347,54.508453,146.094711,376.377835,926.435019,2147.499571,4717.270683,9966.759379,0


In [29]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["label"])

In [30]:
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data

,0,1,2,3,4,5,6,7,8,9,...,302,303,304,305,306,307,308,309,label,subset
0,0.098729,0.031973,0.000365,-0.000007,0.000316,2.256631,5.209220e-07,99.485067,2896.967370,-8.899776,...,22.795197,62.616585,167.777223,432.209379,1063.099120,2466.916935,5421.217569,11453.470216,0,train
1,1.199340,0.750596,0.008740,-0.000003,0.008520,50.971257,1.700527e-03,238.051871,1743.108419,-225.589702,...,17.974283,48.824629,131.831690,338.891959,852.284589,1882.834866,4139.876267,8744.339269,0,train
2,0.065055,0.035480,0.000407,0.000002,0.000384,0.884249,4.524265e-07,37.531094,1085.289724,-5.836638,...,19.023053,52.170481,139.759913,359.944134,886.929095,2056.811512,4520.191658,9549.511321,1,train
3,0.020909,0.017517,0.000200,0.000005,0.000263,0.582813,3.014689e-07,15.647921,483.772550,-0.603839,...,16.141156,44.307164,118.679808,305.529409,754.057437,1749.122918,3841.016648,8109.374258,1,train
4,0.112365,0.026687,0.000302,-0.000005,0.000347,3.670160,5.937945e-07,182.204088,5346.021577,-22.110172,...,25.317232,69.525113,186.272153,479.974834,1180.562208,2738.226881,6014.783419,12715.479526,1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,0.116214,0.070546,0.000837,0.000003,0.000333,1.890006,9.844257e-07,34.083311,896.672947,-10.446397,...,18.210931,50.085484,134.123291,345.396264,850.942761,1973.383824,4336.099395,9158.984652,0,test
122,0.102309,0.051000,0.000506,-0.000016,0.001034,1.847553,1.924126e-05,40.452038,1075.453666,-6.751823,...,19.542231,53.362734,143.150599,368.374046,906.401758,2104.165424,4621.862895,9771.005026,1,test
123,2.069748,0.778888,0.009144,0.000007,0.006748,14.535060,5.509018e-04,495.724029,3238.766974,-624.951034,...,22.791668,62.261803,165.025565,421.498928,1043.915282,2411.579042,5296.857549,11183.090268,0,test
124,0.077996,0.035679,0.000413,-0.000002,0.000190,2.975330,4.895139e-07,49.020883,1472.570512,-3.569180,...,20.194182,55.434210,148.532297,382.323028,939.260746,2182.546624,4796.891458,10136.174429,1,test


In [31]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('float64'), dtype('int64')], dtype=object),
 label
 0    0.666667
 1    0.333333
 Name: proportion, dtype: float64,
 subset
 train    0.698413
 test     0.301587
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.657895
         1        0.342105
 train   0        0.670455
         1        0.329545
 Name: proportion, dtype: float64,
 np.int64(0))

In [35]:
data.to_parquet("../Benchmark/lsvt.parquet", index=False)

# Madelon

In [3]:
input_train_data = pd.read_csv("../Datasets/madelon/MADELON/madelon_train.data", sep=" ", header=None).dropna(axis=1, how="all")
output_train_data = pd.read_csv("../Datasets/madelon/MADELON/madelon_train.labels", sep=" ", header=None, names=["label"])
train_data = pd.concat([input_train_data, output_train_data], axis=1).reset_index(drop=True)
train_data.loc[train_data["label"] == -1, "label"] = 0
train_data["subset"] = "train"
train_data

,0,1,2,3,4,5,6,7,8,9,...,492,493,494,495,496,497,498,499,label,subset
0,485,477,537,479,452,471,491,476,475,473,...,477,485,511,485,481,479,475,496,0,train
1,483,458,460,487,587,475,526,479,485,469,...,487,338,513,486,483,492,510,517,0,train
2,487,542,499,468,448,471,442,478,480,477,...,492,650,506,501,480,489,499,498,0,train
3,480,491,510,485,495,472,417,474,502,476,...,474,572,454,469,475,482,494,461,1,train
4,484,502,528,489,466,481,402,478,487,468,...,452,435,486,508,481,504,495,511,1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,490,505,503,474,463,461,519,476,518,467,...,449,588,499,506,475,463,507,501,1,train
1996,480,475,476,480,495,482,515,479,480,484,...,473,424,454,570,476,493,465,485,0,train
1997,480,517,631,470,485,474,535,476,493,466,...,479,687,488,488,483,500,523,481,0,train
1998,484,481,505,478,542,477,518,477,510,472,...,526,750,486,529,484,473,527,485,1,train


In [6]:
input_test_data = pd.read_csv("../Datasets/madelon/MADELON/madelon_valid.data", sep=" ", header=None).dropna(axis=1, how="all")
output_test_data = pd.read_csv("../Datasets/madelon/MADELON/madelon_valid.labels", sep=" ", header=None, names=["label"])
test_data = pd.concat([input_test_data, output_test_data], axis=1).reset_index(drop=True)
test_data.loc[test_data["label"] == -1, "label"] = 0
test_data["subset"] = "test"
test_data

,0,1,2,3,4,5,6,7,8,9,...,492,493,494,495,496,497,498,499,label,subset
0,483,454,513,495,523,469,453,477,506,479,...,543,259,413,520,485,498,523,510,0,test
1,485,508,493,487,478,472,504,476,479,475,...,535,534,514,452,484,495,548,477,0,test
2,483,521,507,475,493,486,421,475,496,483,...,498,495,508,528,486,465,508,503,0,test
3,474,504,576,480,553,483,524,478,483,483,...,470,463,509,525,479,467,552,517,1,test
4,495,474,523,479,495,488,485,476,497,478,...,522,343,509,520,475,493,506,491,0,test
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,493,458,503,478,517,479,472,478,444,477,...,485,443,517,486,474,489,506,506,1,test
596,481,484,481,490,449,481,467,478,469,483,...,508,599,498,527,481,490,455,451,1,test
597,485,485,530,480,444,487,462,475,509,494,...,502,368,453,482,478,481,484,517,1,test
598,477,469,528,485,483,469,482,477,494,476,...,453,638,471,538,470,490,613,492,1,test


In [10]:
data = pd.concat([train_data, test_data], axis=0, ignore_index=True)
data

,0,1,2,3,4,5,6,7,8,9,...,492,493,494,495,496,497,498,499,label,subset
0,485,477,537,479,452,471,491,476,475,473,...,477,485,511,485,481,479,475,496,0,train
1,483,458,460,487,587,475,526,479,485,469,...,487,338,513,486,483,492,510,517,0,train
2,487,542,499,468,448,471,442,478,480,477,...,492,650,506,501,480,489,499,498,0,train
3,480,491,510,485,495,472,417,474,502,476,...,474,572,454,469,475,482,494,461,1,train
4,484,502,528,489,466,481,402,478,487,468,...,452,435,486,508,481,504,495,511,1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2595,493,458,503,478,517,479,472,478,444,477,...,485,443,517,486,474,489,506,506,1,test
2596,481,484,481,490,449,481,467,478,469,483,...,508,599,498,527,481,490,455,451,1,test
2597,485,485,530,480,444,487,462,475,509,494,...,502,368,453,482,478,481,484,517,1,test
2598,477,469,528,485,483,469,482,477,494,476,...,453,638,471,538,470,490,613,492,1,test


In [11]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('int64')], dtype=object),
 label
 0    0.5
 1    0.5
 Name: proportion, dtype: float64,
 subset
 train    0.769231
 test     0.230769
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.5
         1        0.5
 train   0        0.5
         1        0.5
 Name: proportion, dtype: float64,
 np.int64(0))

In [12]:
data.to_parquet("../Benchmark/madelon.parquet", index=False)

# Dota 2

In [3]:
train_data = pd.read_csv("../Datasets/Dota2_Games_Results/dota2Train.csv", header=None)
train_data.rename(columns={0: "label"}, inplace=True)
train_data.loc[train_data["label"] == -1, "label"] = 0
train_data

,label,1,2,3,4,5,6,7,8,9,...,107,108,109,110,111,112,113,114,115,116
0,0,223,2,2,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,152,2,2,0,0,0,1,0,-1,...,0,0,0,0,0,0,0,0,0,0
2,1,131,2,2,0,0,0,1,0,-1,...,0,0,0,0,0,0,0,0,0,0
3,1,154,2,2,0,0,0,0,0,0,...,-1,0,0,0,0,0,0,0,0,0
4,0,171,2,3,0,0,0,0,0,-1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92645,0,154,2,3,1,0,0,-1,0,0,...,0,0,0,0,0,0,0,0,0,0
92646,1,154,2,2,0,0,0,0,-1,0,...,1,0,0,0,0,0,0,0,0,0
92647,1,111,2,3,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
92648,0,185,2,2,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [4]:
test_data = pd.read_csv("../Datasets/Dota2_Games_Results/dota2Test.csv", header=None)
test_data.rename(columns={0: "label"}, inplace=True)
test_data.loc[test_data["label"] == -1, "label"] = 0
test_data

,label,1,2,3,4,5,6,7,8,9,...,107,108,109,110,111,112,113,114,115,116
0,0,223,8,2,0,-1,0,0,0,0,...,-1,0,0,0,0,0,0,0,0,0
1,1,227,8,2,0,0,0,0,0,0,...,-1,0,0,0,0,0,0,0,0,0
2,0,136,2,2,1,0,0,0,-1,0,...,0,0,0,0,0,0,0,0,0,0
3,1,227,2,2,-1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,184,2,3,0,0,0,-1,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10289,1,121,2,2,0,0,0,0,0,0,...,0,-1,0,0,0,0,0,0,0,0
10290,1,154,9,2,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
10291,1,122,9,2,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
10292,1,152,2,3,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
train_data.columns = [["label"] + list(range(len(train_data.columns))[:-1])]
train_data = train_data[list(range(len(train_data.columns))[:-1]) + ["label"]].copy()
train_data.columns = [col[0] if isinstance(col, tuple) else col for col in train_data.columns]

In [6]:
test_data.columns = [["label"] + list(range(len(test_data.columns))[:-1])]
test_data = test_data[list(range(len(test_data.columns))[:-1]) + ["label"]].copy()
test_data.columns = [col[0] if isinstance(col, tuple) else col for col in test_data.columns]

In [7]:
train_data.dtypes.unique(), test_data.dtypes.unique()

(array([dtype('int64')], dtype=object), array([dtype('int64')], dtype=object))

In [8]:
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data

,0,1,2,3,4,5,6,7,8,9,...,108,109,110,111,112,113,114,115,label,subset
0,223,2,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,train
1,152,2,2,0,0,0,1,0,-1,0,...,0,0,0,0,0,0,0,0,1,train
2,131,2,2,0,0,0,1,0,-1,0,...,0,0,0,0,0,0,0,0,1,train
3,154,2,2,0,0,0,0,0,0,-1,...,0,0,0,0,0,0,0,0,1,train
4,171,2,3,0,0,0,0,0,-1,0,...,0,0,0,0,0,0,0,0,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102939,121,2,2,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1,test
102940,154,9,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,test
102941,122,9,2,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,1,test
102942,152,2,3,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,test


In [9]:
data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(label
 1    0.527316
 0    0.472684
 Name: proportion, dtype: float64,
 subset
 train    0.900004
 test     0.099996
 Name: proportion, dtype: float64,
 subset  label
 test    1        0.534486
         0        0.465514
 train   1        0.526519
         0        0.473481
 Name: proportion, dtype: float64,
 np.int64(0))

In [10]:
data.to_parquet("../Benchmark/dota2.parquet", index=False)

# Musk (Version 1)

In [11]:
# Read and decompress the .Z file
with open("../Datasets/Musk_Version_1/clean1.data.Z", "rb") as f:
    data = unlzw(f.read())

# Convert bytes → string → DataFrame
data = pd.read_csv(StringIO(data.decode("utf-8")), header=None)

data

,0,1,2,3,4,5,6,7,8,9,...,159,160,161,162,163,164,165,166,167,168
0,MUSK-188,188_1+1,42,-198,-109,-75,-117,11,23,-88,...,-74,-129,-120,-38,30,48,-37,6,30,1.0
1,MUSK-188,188_1+2,42,-191,-142,-65,-117,55,49,-170,...,-302,60,-120,-39,31,48,-37,5,30,1.0
2,MUSK-188,188_1+3,42,-191,-142,-75,-117,11,49,-161,...,-73,-127,-120,-38,30,48,-37,5,31,1.0
3,MUSK-188,188_1+4,42,-198,-110,-65,-117,55,23,-95,...,-302,60,-120,-39,30,48,-37,6,30,1.0
4,MUSK-190,190_1+1,42,-198,-102,-75,-117,10,24,-87,...,-73,-127,51,128,144,43,-30,14,26,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
471,NON-MUSK-jp13,jp13_1+4,49,-199,-161,29,-95,-86,-48,2,...,-246,-209,33,152,134,47,-43,-15,-10,0.0
472,NON-MUSK-jp13,jp13_2+1,38,-123,-139,30,-117,-88,214,-13,...,-226,-210,20,55,119,79,-28,4,74,0.0
473,NON-MUSK-jp13,jp13_2+2,43,-102,-20,-101,-116,200,-166,66,...,32,136,-15,143,121,55,-37,-19,-36,0.0
474,NON-MUSK-jp13,jp13_2+3,39,-58,27,31,-117,-92,85,21,...,-232,-206,13,45,116,79,-28,3,74,0.0


In [12]:
feature_names = []

with open("../Datasets/Musk_Version_1/clean1.names", "r") as f:
    for line in f:
        line = line.strip()
        if ":" in line:
            feature = line.split(":", 1)[0].strip()
            feature_names.append(feature)

print(len(feature_names), "features found")
print(feature_names[:10])  # preview

168 features found
['molecule_name', 'conformation_name', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8']


In [13]:
data.columns = feature_names + ["label"]
data["label"] = data["label"].astype(int)

In [14]:
data

,molecule_name,conformation_name,f1,f2,f3,f4,f5,f6,f7,f8,...,f158,f159,f160,f161,f162,f163,f164,f165,f166,label
0,MUSK-188,188_1+1,42,-198,-109,-75,-117,11,23,-88,...,-74,-129,-120,-38,30,48,-37,6,30,1
1,MUSK-188,188_1+2,42,-191,-142,-65,-117,55,49,-170,...,-302,60,-120,-39,31,48,-37,5,30,1
2,MUSK-188,188_1+3,42,-191,-142,-75,-117,11,49,-161,...,-73,-127,-120,-38,30,48,-37,5,31,1
3,MUSK-188,188_1+4,42,-198,-110,-65,-117,55,23,-95,...,-302,60,-120,-39,30,48,-37,6,30,1
4,MUSK-190,190_1+1,42,-198,-102,-75,-117,10,24,-87,...,-73,-127,51,128,144,43,-30,14,26,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
471,NON-MUSK-jp13,jp13_1+4,49,-199,-161,29,-95,-86,-48,2,...,-246,-209,33,152,134,47,-43,-15,-10,0
472,NON-MUSK-jp13,jp13_2+1,38,-123,-139,30,-117,-88,214,-13,...,-226,-210,20,55,119,79,-28,4,74,0
473,NON-MUSK-jp13,jp13_2+2,43,-102,-20,-101,-116,200,-166,66,...,32,136,-15,143,121,55,-37,-19,-36,0
474,NON-MUSK-jp13,jp13_2+3,39,-58,27,31,-117,-92,85,21,...,-232,-206,13,45,116,79,-28,3,74,0


In [15]:
# Get each molecule and its label (assuming all conformations share the same label)
molecule_labels = data.groupby("molecule_name")["label"].first().reset_index()

# Stratified split by label, not by individual rows
train_mols, test_mols = train_test_split(
    molecule_labels["molecule_name"],
    test_size=TEST_SIZE,
    stratify=molecule_labels["label"],
    random_state=RANDOM_STATE
)

# Select all rows that belong to those molecules
train_df = data[data["molecule_name"].isin(train_mols)].copy()
test_df = data[data["molecule_name"].isin(test_mols)].copy()

print(f"Train molecules: {len(train_mols)}, Test molecules: {len(test_mols)}")
print(f"Train rows: {len(train_df)}, Test rows: {len(test_df)}")

Train molecules: 64, Test molecules: 28
Train rows: 344, Test rows: 132


In [16]:
set(train_df["molecule_name"]).intersection(set(test_df["molecule_name"])), set(train_df["conformation_name"]).intersection(set(test_df["conformation_name"]))

(set(), set())

In [17]:
train_df["subset"] = "train"
test_df["subset"] = "test"
data = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)
data

,molecule_name,conformation_name,f1,f2,f3,f4,f5,f6,f7,f8,...,f159,f160,f161,f162,f163,f164,f165,f166,label,subset
0,MUSK-188,188_1+1,42,-198,-109,-75,-117,11,23,-88,...,-129,-120,-38,30,48,-37,6,30,1,train
1,MUSK-188,188_1+2,42,-191,-142,-65,-117,55,49,-170,...,60,-120,-39,31,48,-37,5,30,1,train
2,MUSK-188,188_1+3,42,-191,-142,-75,-117,11,49,-161,...,-127,-120,-38,30,48,-37,5,31,1,train
3,MUSK-188,188_1+4,42,-198,-110,-65,-117,55,23,-95,...,60,-120,-39,30,48,-37,6,30,1,train
4,MUSK-211,211_1+1,40,-173,-142,13,-116,-7,50,-171,...,20,38,88,133,66,-28,13,58,1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
471,NON-MUSK-jp13,jp13_1+4,49,-199,-161,29,-95,-86,-48,2,...,-209,33,152,134,47,-43,-15,-10,0,test
472,NON-MUSK-jp13,jp13_2+1,38,-123,-139,30,-117,-88,214,-13,...,-210,20,55,119,79,-28,4,74,0,test
473,NON-MUSK-jp13,jp13_2+2,43,-102,-20,-101,-116,200,-166,66,...,136,-15,143,121,55,-37,-19,-36,0,test
474,NON-MUSK-jp13,jp13_2+3,39,-58,27,31,-117,-92,85,21,...,-206,13,45,116,79,-28,3,74,0,test


In [18]:
data.drop(columns=["molecule_name", "conformation_name"], inplace=True)

In [19]:
data.drop(columns=["subset"]).dtypes.unique()

array([dtype('int64')], dtype=object)

In [20]:
data.columns = [list(range(len(data.columns))[:-2]) + ["label", "subset"]]
data = data[list(range(len(data.columns))[:-2]) + ["label", "subset"]].copy()

In [21]:
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]

In [22]:
data

,0,1,2,3,4,5,6,7,8,9,...,158,159,160,161,162,163,164,165,label,subset
0,42,-198,-109,-75,-117,11,23,-88,-28,-27,...,-129,-120,-38,30,48,-37,6,30,1,train
1,42,-191,-142,-65,-117,55,49,-170,-45,5,...,60,-120,-39,31,48,-37,5,30,1,train
2,42,-191,-142,-75,-117,11,49,-161,-45,-28,...,-127,-120,-38,30,48,-37,5,31,1,train
3,42,-198,-110,-65,-117,55,23,-95,-28,5,...,60,-120,-39,30,48,-37,6,30,1,train
4,40,-173,-142,13,-116,-7,50,-171,-44,-103,...,20,38,88,133,66,-28,13,58,1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
471,49,-199,-161,29,-95,-86,-48,2,112,-79,...,-209,33,152,134,47,-43,-15,-10,0,test
472,38,-123,-139,30,-117,-88,214,-13,-74,-129,...,-210,20,55,119,79,-28,4,74,0,test
473,43,-102,-20,-101,-116,200,-166,66,-222,-49,...,136,-15,143,121,55,-37,-19,-36,0,test
474,39,-58,27,31,-117,-92,85,21,-73,-68,...,-206,13,45,116,79,-28,3,74,0,test


In [ ]:
data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(label
 1    0.746032
 0    0.253968
 Name: proportion, dtype: float64,
 subset
 train    0.698413
 test     0.301587
 Name: proportion, dtype: float64,
 subset  label
 test    1        0.750000
         0        0.250000
 train   1        0.744318
         0        0.255682
 Name: proportion, dtype: float64,
 np.int64(0))

In [24]:
data.to_parquet("../Benchmark/musk_version_1.parquet", index=False)

# Drug Induced Autoimmunity Prediction

In [25]:
train_data = pd.read_csv("../Datasets/Drug_Induced_Autoimmunity_Prediction/DIA_trainingset_RDKit_descriptors.csv")
train_data

,Label,SMILES,BalabanJ,BertzCT,Chi0,Chi0n,Chi0v,Chi1,Chi1n,Chi1v,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,0,COC(=O)N(C)c1c(N)nc(nc1N)c2nn(Cc3ccccc3F)c4ncc...,1.821,1266.407,22.121,16.781,16.781,14.901,9.203,9.203,...,0,0,0,0,0,0,0,0,0,0
1,0,C[C@H](N(O)C(=O)N)c1cc2ccccc2s1,2.363,490.434,11.707,8.752,9.569,7.592,4.854,5.670,...,0,0,0,0,0,0,0,1,0,1
2,0,C[N+](C)(C)CC(=O)[O-],3.551,93.092,6.784,5.471,5.471,3.417,2.420,2.420,...,0,0,0,0,0,0,0,0,0,0
3,1,CC(C)n1c(\C=C\[C@H](O)C[C@H](O)CC(=O)O)c(c2ccc...,2.076,1053.003,21.836,16.995,16.995,14.274,9.926,9.926,...,0,0,0,0,0,0,0,0,0,0
4,1,C\C(=C(\C#N)/C(=O)Nc1ccc(cc1)C(F)(F)F)\O,2.888,549.823,14.629,9.746,9.746,8.752,5.040,5.040,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
472,0,C(C1=NCCN1)c2cccc3ccccc23,2.022,537.932,10.795,9.110,9.110,7.933,5.672,5.672,...,0,0,0,0,0,0,0,0,0,0
473,0,C[N@+]1(CC2CC2)CC[C@]34[C@H]5Oc6c(O)ccc(C[C@@H...,1.602,848.658,17.897,15.202,15.202,12.389,10.003,10.003,...,0,0,0,0,0,0,0,0,0,0
474,1,CO\N=C(/C(=O)N[C@H]1[C@H]2SCC(=C(N2C1=O)C(=O)O...,1.766,910.031,21.129,14.986,15.802,13.845,8.129,9.178,...,1,0,0,0,0,0,0,0,0,0
475,0,Clc1ccc(CO\N=C(\Cn2ccnc2)/c3ccc(Cl)cc3Cl)c(Cl)c1,1.831,926.191,18.518,13.372,16.396,12.525,7.566,9.078,...,0,0,0,0,0,0,0,0,0,0


In [26]:
test_data = pd.read_csv("../Datasets/Drug_Induced_Autoimmunity_Prediction/DIA_testset_RDKit_descriptors.csv")
test_data

,Label,SMILES,BalabanJ,BertzCT,Chi0,Chi0n,Chi0v,Chi1,Chi1n,Chi1v,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,0,C[C@H](\C=C\[C@H](O)C1CC1)[C@@H]2CC[C@@H]3\C(=...,1.484,743.207,21.466,18.764,18.764,14.292,12.106,12.106,...,0,0,0,0,0,0,0,0,0,0
1,1,OCCN1CCN(CCCN2c3ccccc3Sc4ccc(cc24)C(F)(F)F)CC1,1.472,868.947,21.140,16.736,17.553,14.453,10.268,11.084,...,0,0,0,0,0,0,0,0,0,0
2,0,C[C@@H]1O[C@H](C[C@H](O)[C@@H]1O)O[C@@H]2[C@H]...,0.837,1409.004,39.189,32.904,32.904,26.011,20.941,20.941,...,0,0,0,0,0,0,0,0,0,0
3,1,NC(=O)Cc1cccc(C(=O)c2ccccc2)c1N,2.406,621.298,13.828,10.297,10.297,9.092,5.847,5.847,...,0,0,0,0,0,0,0,0,0,0
4,0,COc1cc2c(CCN[C@]23CS[C@@H]4[C@@H]5[C@@H]6N(C)[...,1.320,2127.996,37.955,30.849,31.666,25.910,18.066,19.115,...,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,0,CCN1CCN(C(=O)N[C@H](C(=O)N[C@@H]2[C@@H]3SC(C)(...,1.508,1127.109,26.361,19.925,20.742,16.973,11.450,12.330,...,1,0,0,0,0,0,0,0,0,1
116,0,CC1=C(C=C(C#N)C(=O)N1)c2ccncc2,2.678,608.396,11.544,8.689,8.689,7.720,4.765,4.765,...,0,0,0,0,0,0,0,0,0,0
117,0,CCCN(CCc1cccs1)[C@@H]2CCc3c(O)cccc3C2,1.670,593.488,15.364,13.294,14.110,10.775,8.338,9.217,...,0,0,0,0,0,0,0,1,0,0
118,0,COCCOC(=O)C1=C(C)NC(=C([C@@H]1c2cccc(c2)[N+](=...,2.603,902.371,22.422,17.683,17.683,14.167,9.469,9.469,...,0,0,0,0,0,0,0,0,1,0


In [27]:
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data

,Label,SMILES,BalabanJ,BertzCT,Chi0,Chi0n,Chi0v,Chi1,Chi1n,Chi1v,...,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea,subset
0,0,COC(=O)N(C)c1c(N)nc(nc1N)c2nn(Cc3ccccc3F)c4ncc...,1.821,1266.407,22.121,16.781,16.781,14.901,9.203,9.203,...,0,0,0,0,0,0,0,0,0,train
1,0,C[C@H](N(O)C(=O)N)c1cc2ccccc2s1,2.363,490.434,11.707,8.752,9.569,7.592,4.854,5.670,...,0,0,0,0,0,0,1,0,1,train
2,0,C[N+](C)(C)CC(=O)[O-],3.551,93.092,6.784,5.471,5.471,3.417,2.420,2.420,...,0,0,0,0,0,0,0,0,0,train
3,1,CC(C)n1c(\C=C\[C@H](O)C[C@H](O)CC(=O)O)c(c2ccc...,2.076,1053.003,21.836,16.995,16.995,14.274,9.926,9.926,...,0,0,0,0,0,0,0,0,0,train
4,1,C\C(=C(\C#N)/C(=O)Nc1ccc(cc1)C(F)(F)F)\O,2.888,549.823,14.629,9.746,9.746,8.752,5.040,5.040,...,0,0,0,0,0,0,0,0,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
592,0,CCN1CCN(C(=O)N[C@H](C(=O)N[C@@H]2[C@@H]3SC(C)(...,1.508,1127.109,26.361,19.925,20.742,16.973,11.450,12.330,...,0,0,0,0,0,0,0,0,1,test
593,0,CC1=C(C=C(C#N)C(=O)N1)c2ccncc2,2.678,608.396,11.544,8.689,8.689,7.720,4.765,4.765,...,0,0,0,0,0,0,0,0,0,test
594,0,CCCN(CCc1cccs1)[C@@H]2CCc3c(O)cccc3C2,1.670,593.488,15.364,13.294,14.110,10.775,8.338,9.217,...,0,0,0,0,0,0,1,0,0,test
595,0,COCCOC(=O)C1=C(C)NC(=C([C@@H]1c2cccc(c2)[N+](=...,2.603,902.371,22.422,17.683,17.683,14.167,9.469,9.469,...,0,0,0,0,0,0,0,1,0,test


In [28]:
data["SMILES"].nunique()

597

In [29]:
data.rename(columns={"Label": "label"}, inplace=True)
data.drop(columns=["SMILES"], inplace=True)

In [30]:
data.drop(columns=["subset"]).dtypes.unique()

array([dtype('int64'), dtype('float64')], dtype=object)

In [31]:
data

,label,BalabanJ,BertzCT,Chi0,Chi0n,Chi0v,Chi1,Chi1n,Chi1v,Chi2n,...,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea,subset
0,0,1.821,1266.407,22.121,16.781,16.781,14.901,9.203,9.203,6.668,...,0,0,0,0,0,0,0,0,0,train
1,0,2.363,490.434,11.707,8.752,9.569,7.592,4.854,5.670,3.545,...,0,0,0,0,0,0,1,0,1,train
2,0,3.551,93.092,6.784,5.471,5.471,3.417,2.420,2.420,2.820,...,0,0,0,0,0,0,0,0,0,train
3,1,2.076,1053.003,21.836,16.995,16.995,14.274,9.926,9.926,7.662,...,0,0,0,0,0,0,0,0,0,train
4,1,2.888,549.823,14.629,9.746,9.746,8.752,5.040,5.040,3.601,...,0,0,0,0,0,0,0,0,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
592,0,1.508,1127.109,26.361,19.925,20.742,16.973,11.450,12.330,9.200,...,0,0,0,0,0,0,0,0,1,test
593,0,2.678,608.396,11.544,8.689,8.689,7.720,4.765,4.765,3.399,...,0,0,0,0,0,0,0,0,0,test
594,0,1.670,593.488,15.364,13.294,14.110,10.775,8.338,9.217,6.177,...,0,0,0,0,0,0,1,0,0,test
595,0,2.603,902.371,22.422,17.683,17.683,14.167,9.469,9.469,7.070,...,0,0,0,0,0,0,0,1,0,test


In [32]:
feat_cols = [col for col in data.columns if col != "label" and col != "subset"]
data = data[feat_cols + ["label", "subset"]].copy()
data

,BalabanJ,BertzCT,Chi0,Chi0n,Chi0v,Chi1,Chi1n,Chi1v,Chi2n,Chi2v,...,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea,label,subset
0,1.821,1266.407,22.121,16.781,16.781,14.901,9.203,9.203,6.668,6.668,...,0,0,0,0,0,0,0,0,0,train
1,2.363,490.434,11.707,8.752,9.569,7.592,4.854,5.670,3.545,4.661,...,0,0,0,0,0,1,0,1,0,train
2,3.551,93.092,6.784,5.471,5.471,3.417,2.420,2.420,2.820,2.820,...,0,0,0,0,0,0,0,0,0,train
3,2.076,1053.003,21.836,16.995,16.995,14.274,9.926,9.926,7.662,7.662,...,0,0,0,0,0,0,0,0,1,train
4,2.888,549.823,14.629,9.746,9.746,8.752,5.040,5.040,3.601,3.601,...,0,0,0,0,0,0,0,0,1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
592,1.508,1127.109,26.361,19.925,20.742,16.973,11.450,12.330,9.200,10.971,...,0,0,0,0,0,0,0,1,0,test
593,2.678,608.396,11.544,8.689,8.689,7.720,4.765,4.765,3.399,3.399,...,0,0,0,0,0,0,0,0,0,test
594,1.670,593.488,15.364,13.294,14.110,10.775,8.338,9.217,6.177,7.209,...,0,0,0,0,0,1,0,0,0,test
595,2.603,902.371,22.422,17.683,17.683,14.167,9.469,9.469,7.070,7.070,...,0,0,0,0,0,0,1,0,0,test


In [33]:
data.columns = [list(range(len(data.columns))[:-2]) + ["label", "subset"]]
data = data[list(range(len(data.columns))[:-2]) + ["label", "subset"]].copy()
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]

In [34]:
data

,0,1,2,3,4,5,6,7,8,9,...,188,189,190,191,192,193,194,195,label,subset
0,1.821,1266.407,22.121,16.781,16.781,14.901,9.203,9.203,6.668,6.668,...,0,0,0,0,0,0,0,0,0,train
1,2.363,490.434,11.707,8.752,9.569,7.592,4.854,5.670,3.545,4.661,...,0,0,0,0,0,1,0,1,0,train
2,3.551,93.092,6.784,5.471,5.471,3.417,2.420,2.420,2.820,2.820,...,0,0,0,0,0,0,0,0,0,train
3,2.076,1053.003,21.836,16.995,16.995,14.274,9.926,9.926,7.662,7.662,...,0,0,0,0,0,0,0,0,1,train
4,2.888,549.823,14.629,9.746,9.746,8.752,5.040,5.040,3.601,3.601,...,0,0,0,0,0,0,0,0,1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
592,1.508,1127.109,26.361,19.925,20.742,16.973,11.450,12.330,9.200,10.971,...,0,0,0,0,0,0,0,1,0,test
593,2.678,608.396,11.544,8.689,8.689,7.720,4.765,4.765,3.399,3.399,...,0,0,0,0,0,0,0,0,0,test
594,1.670,593.488,15.364,13.294,14.110,10.775,8.338,9.217,6.177,7.209,...,0,0,0,0,0,1,0,0,0,test
595,2.603,902.371,22.422,17.683,17.683,14.167,9.469,9.469,7.070,7.070,...,0,0,0,0,0,0,1,0,0,test


In [35]:
data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(label
 0    0.752094
 1    0.247906
 Name: proportion, dtype: float64,
 subset
 train    0.798995
 test     0.201005
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.750000
         1        0.250000
 train   0        0.752621
         1        0.247379
 Name: proportion, dtype: float64,
 np.int64(0))

In [36]:
data.to_parquet("../Benchmark/drug_autoimmunity.parquet", index=False)

# TUANDROMD (Tezpur University Android Malware Dataset)

In [37]:
data = pd.read_csv("../Datasets/TUANDROMD_Tezpur_University_Android_Malware_Dataset/TUANDROMD.csv")
data

,ACCESS_ALL_DOWNLOADS,ACCESS_CACHE_FILESYSTEM,ACCESS_CHECKIN_PROPERTIES,ACCESS_COARSE_LOCATION,ACCESS_COARSE_UPDATES,ACCESS_FINE_LOCATION,ACCESS_LOCATION_EXTRA_COMMANDS,ACCESS_MOCK_LOCATION,ACCESS_MTK_MMHW,ACCESS_NETWORK_STATE,...,Landroid/telephony/TelephonyManager;->getLine1Number,Landroid/telephony/TelephonyManager;->getNetworkOperator,Landroid/telephony/TelephonyManager;->getNetworkOperatorName,Landroid/telephony/TelephonyManager;->getNetworkCountryIso,Landroid/telephony/TelephonyManager;->getSimOperator,Landroid/telephony/TelephonyManager;->getSimOperatorName,Landroid/telephony/TelephonyManager;->getSimCountryIso,Landroid/telephony/TelephonyManager;->getSimSerialNumber,Lorg/apache/http/impl/client/DefaultHttpClient;->execute,Label
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,malware
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,malware
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,malware
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,malware
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,malware
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4460,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,goodware
4461,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,goodware
4462,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,goodware
4463,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,goodware


In [38]:
data.iloc[data.index[data.isna().any(axis=1)]]

,ACCESS_ALL_DOWNLOADS,ACCESS_CACHE_FILESYSTEM,ACCESS_CHECKIN_PROPERTIES,ACCESS_COARSE_LOCATION,ACCESS_COARSE_UPDATES,ACCESS_FINE_LOCATION,ACCESS_LOCATION_EXTRA_COMMANDS,ACCESS_MOCK_LOCATION,ACCESS_MTK_MMHW,ACCESS_NETWORK_STATE,...,Landroid/telephony/TelephonyManager;->getLine1Number,Landroid/telephony/TelephonyManager;->getNetworkOperator,Landroid/telephony/TelephonyManager;->getNetworkOperatorName,Landroid/telephony/TelephonyManager;->getNetworkCountryIso,Landroid/telephony/TelephonyManager;->getSimOperator,Landroid/telephony/TelephonyManager;->getSimOperatorName,Landroid/telephony/TelephonyManager;->getSimCountryIso,Landroid/telephony/TelephonyManager;->getSimSerialNumber,Lorg/apache/http/impl/client/DefaultHttpClient;->execute,Label
2533,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
data.dropna(axis=0, inplace=True)

In [40]:
data.rename(columns={"Label": "label"}, inplace=True)
data.loc[data["label"] == "malware", "label"] = 1
data.loc[data["label"] == "goodware", "label"] = 0
data["label"] = data["label"].astype(int)

In [41]:
data

,ACCESS_ALL_DOWNLOADS,ACCESS_CACHE_FILESYSTEM,ACCESS_CHECKIN_PROPERTIES,ACCESS_COARSE_LOCATION,ACCESS_COARSE_UPDATES,ACCESS_FINE_LOCATION,ACCESS_LOCATION_EXTRA_COMMANDS,ACCESS_MOCK_LOCATION,ACCESS_MTK_MMHW,ACCESS_NETWORK_STATE,...,Landroid/telephony/TelephonyManager;->getLine1Number,Landroid/telephony/TelephonyManager;->getNetworkOperator,Landroid/telephony/TelephonyManager;->getNetworkOperatorName,Landroid/telephony/TelephonyManager;->getNetworkCountryIso,Landroid/telephony/TelephonyManager;->getSimOperator,Landroid/telephony/TelephonyManager;->getSimOperatorName,Landroid/telephony/TelephonyManager;->getSimCountryIso,Landroid/telephony/TelephonyManager;->getSimSerialNumber,Lorg/apache/http/impl/client/DefaultHttpClient;->execute,label
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4460,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
4461,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4462,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4463,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [42]:
data.columns = [list(range(len(data.columns))[:-1]) + ["label"]]
data = data[list(range(len(data.columns))[:-1]) + ["label"]].copy()
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]

In [43]:
data

,0,1,2,3,4,5,6,7,8,9,...,232,233,234,235,236,237,238,239,240,label
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4460,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0
4461,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4462,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4463,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [ ]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["label"])

In [45]:
train_data["label"].value_counts(normalize=True), test_data["label"].value_counts(normalize=True)

(label
 1    0.798656
 0    0.201344
 Name: proportion, dtype: float64,
 label
 1    0.798507
 0    0.201493
 Name: proportion, dtype: float64)

In [46]:
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data

,0,1,2,3,4,5,6,7,8,9,...,233,234,235,236,237,238,239,240,label,subset
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,train
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,train
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,train
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1,train
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4459,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,test
4460,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1,test
4461,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1,test
4462,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,test


In [47]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('float64'), dtype('int64')], dtype=object),
 label
 1    0.798611
 0    0.201389
 Name: proportion, dtype: float64,
 subset
 train    0.699821
 test     0.300179
 Name: proportion, dtype: float64,
 subset  label
 test    1        0.798507
         0        0.201493
 train   1        0.798656
         0        0.201344
 Name: proportion, dtype: float64,
 np.int64(0))

In [48]:
data.to_parquet("../Benchmark/tuandromd.parquet", index=False)

# Darwin

In [49]:
data = pd.read_csv("../Datasets/DARWIN/data.csv")
data

,ID,air_time1,disp_index1,gmrt_in_air1,gmrt_on_paper1,max_x_extension1,max_y_extension1,mean_acc_in_air1,mean_acc_on_paper1,mean_gmrt1,...,mean_jerk_in_air25,mean_jerk_on_paper25,mean_speed_in_air25,mean_speed_on_paper25,num_of_pendown25,paper_time25,pressure_mean25,pressure_var25,total_time25,class
0,id_1,5160,0.000013,120.804174,86.853334,957,6601,0.361800,0.217459,103.828754,...,0.141434,0.024471,5.596487,3.184589,71,40120,1749.278166,296102.7676,144605,P
1,id_2,51980,0.000016,115.318238,83.448681,1694,6998,0.272513,0.144880,99.383459,...,0.049663,0.018368,1.665973,0.950249,129,126700,1504.768272,278744.2850,298640,P
2,id_3,2600,0.000010,229.933997,172.761858,2333,5802,0.387020,0.181342,201.347928,...,0.178194,0.017174,4.000781,2.392521,74,45480,1431.443492,144411.7055,79025,P
3,id_4,2130,0.000010,369.403342,183.193104,1756,8159,0.556879,0.164502,276.298223,...,0.113905,0.019860,4.206746,1.613522,123,67945,1465.843329,230184.7154,181220,P
4,id_5,2310,0.000007,257.997131,111.275889,987,4732,0.266077,0.145104,184.636510,...,0.121782,0.020872,3.319036,1.680629,92,37285,1841.702561,158290.0255,72575,P
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169,id_170,2930,0.000010,241.736477,176.115957,1839,6439,0.253347,0.174663,208.926217,...,0.119152,0.020909,4.508709,2.233198,96,44545,1798.923336,247448.3108,80335,H
170,id_171,2140,0.000009,274.728964,234.495802,2053,8487,0.225537,0.174920,254.612383,...,0.174495,0.017640,4.685573,2.806888,84,37560,1725.619941,160664.6464,345835,H
171,id_172,3830,0.000008,151.536989,171.104693,1287,7352,0.165480,0.161058,161.320841,...,0.114472,0.017194,3.493815,2.510601,88,51675,1915.573488,128727.1241,83445,H
172,id_173,1760,0.000008,289.518195,196.411138,1674,6946,0.518937,0.202613,242.964666,...,0.114472,0.017194,3.493815,2.510601,88,51675,1915.573488,128727.1241,83445,H


In [50]:
data["ID"].nunique()

174

In [51]:
data.rename(columns={"class": "label"}, inplace=True)
data.loc[data["label"] == "P", "label"] = 1
data.loc[data["label"] == "H", "label"] = 0
data["label"] = data["label"].astype(int)
data["label"].value_counts(normalize=True)

label
1    0.511494
0    0.488506
Name: proportion, dtype: float64

In [52]:
data.drop(columns=["ID"], inplace=True)
data.columns = [list(range(len(data.columns))[:-1]) + ["label"]]
data = data[list(range(len(data.columns))[:-1]) + ["label"]].copy()
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]

In [53]:
data

,0,1,2,3,4,5,6,7,8,9,...,441,442,443,444,445,446,447,448,449,label
0,5160,0.000013,120.804174,86.853334,957,6601,0.361800,0.217459,103.828754,0.051836,...,0.141434,0.024471,5.596487,3.184589,71,40120,1749.278166,296102.7676,144605,1
1,51980,0.000016,115.318238,83.448681,1694,6998,0.272513,0.144880,99.383459,0.039827,...,0.049663,0.018368,1.665973,0.950249,129,126700,1504.768272,278744.2850,298640,1
2,2600,0.000010,229.933997,172.761858,2333,5802,0.387020,0.181342,201.347928,0.064220,...,0.178194,0.017174,4.000781,2.392521,74,45480,1431.443492,144411.7055,79025,1
3,2130,0.000010,369.403342,183.193104,1756,8159,0.556879,0.164502,276.298223,0.090408,...,0.113905,0.019860,4.206746,1.613522,123,67945,1465.843329,230184.7154,181220,1
4,2310,0.000007,257.997131,111.275889,987,4732,0.266077,0.145104,184.636510,0.037528,...,0.121782,0.020872,3.319036,1.680629,92,37285,1841.702561,158290.0255,72575,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169,2930,0.000010,241.736477,176.115957,1839,6439,0.253347,0.174663,208.926217,0.032691,...,0.119152,0.020909,4.508709,2.233198,96,44545,1798.923336,247448.3108,80335,0
170,2140,0.000009,274.728964,234.495802,2053,8487,0.225537,0.174920,254.612383,0.032059,...,0.174495,0.017640,4.685573,2.806888,84,37560,1725.619941,160664.6464,345835,0
171,3830,0.000008,151.536989,171.104693,1287,7352,0.165480,0.161058,161.320841,0.022705,...,0.114472,0.017194,3.493815,2.510601,88,51675,1915.573488,128727.1241,83445,0
172,1760,0.000008,289.518195,196.411138,1674,6946,0.518937,0.202613,242.964666,0.090686,...,0.114472,0.017194,3.493815,2.510601,88,51675,1915.573488,128727.1241,83445,0


In [ ]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["label"])

In [55]:
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data

,0,1,2,3,4,5,6,7,8,9,...,442,443,444,445,446,447,448,449,label,subset
0,1870,0.000006,595.572014,498.890817,2316,8467,1.043333,0.313464,547.231416,0.190816,...,0.020917,3.709496,2.166313,124,38330,1739.271067,264041.37160,77215,0,train
1,5975,0.000005,468.028309,254.541305,1443,6013,2.772566,0.170472,361.284807,0.543199,...,0.020231,4.298868,2.999059,72,30395,1727.875637,187862.81990,50730,0,train
2,7770,0.000007,252.920870,241.011868,2076,5683,0.498263,0.193680,246.966369,0.084725,...,0.020821,6.278983,3.277159,58,33560,1840.851162,144392.44640,63130,0,train
3,65,0.000006,875.811029,278.542889,2224,4606,0.949719,0.225705,577.176959,0.037820,...,0.023427,4.963550,5.078727,128,39750,1782.381006,210949.70930,87250,0,train
4,5770,0.000008,131.999690,86.784455,1208,6298,0.383622,0.156398,109.392073,0.060444,...,0.020357,3.566246,2.595331,97,40385,1497.226693,175888.77400,82210,1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169,4115,0.000008,177.294869,103.118202,1257,4903,0.570892,0.160674,140.206535,0.100500,...,0.017024,3.351319,2.769629,82,33765,626.638087,73347.34392,81715,1,test
170,1730,0.000008,230.195295,154.737220,1627,6630,0.219523,0.134745,192.466258,0.030817,...,0.018480,4.948004,2.211455,84,40685,1722.652206,213407.52290,81860,0,test
171,1970,0.000011,231.499777,90.649480,1434,5643,0.209560,0.144054,161.074629,0.031652,...,0.017974,2.022657,1.528793,104,71795,912.940525,145022.11890,182355,1,test
172,2105,0.000016,436.403088,254.811203,1759,13493,0.478393,0.175589,345.607145,0.082898,...,0.018896,4.806361,3.335828,60,35920,1929.296214,94373.02807,59175,1,test


In [56]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('int64'), dtype('float64')], dtype=object),
 label
 1    0.511494
 0    0.488506
 Name: proportion, dtype: float64,
 subset
 train    0.695402
 test     0.304598
 Name: proportion, dtype: float64,
 subset  label
 test    1        0.509434
         0        0.490566
 train   1        0.512397
         0        0.487603
 Name: proportion, dtype: float64,
 np.int64(0))

In [57]:
data.to_parquet("../Benchmark/darwin.parquet", index=False)

# QSAR oral toxicity

In [58]:
data = pd.read_csv("../Datasets/QSAR_oral_toxicity/qsar_oral_toxicity.csv", sep=";", header=None)
data

,0,1,2,3,4,5,6,7,8,9,...,1015,1016,1017,1018,1019,1020,1021,1022,1023,1024
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,negative
1,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,negative
2,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,negative
3,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,negative
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,negative
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8987,0,0,0,0,0,0,0,1,0,0,...,0,0,0,1,0,0,0,0,0,negative
8988,0,1,0,0,0,1,0,1,0,0,...,0,0,0,1,0,0,0,0,0,negative
8989,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,1,0,negative
8990,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,negative


In [59]:
data.rename(columns={1024: "label"}, inplace=True)

In [60]:
data.loc[data["label"] == "negative", "label"] = 0
data.loc[data["label"] == "positive", "label"] = 1
data["label"] = data["label"].astype(int)
data["label"].value_counts(normalize=True)

label
0    0.917593
1    0.082407
Name: proportion, dtype: float64

In [ ]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["label"])

In [62]:
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data

,0,1,2,3,4,5,6,7,8,9,...,1016,1017,1018,1019,1020,1021,1022,1023,label,subset
0,0,0,0,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,train
1,0,0,0,1,0,0,0,1,0,0,...,0,1,0,0,0,0,0,0,0,train
2,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,train
3,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,train
4,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8987,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,test
8988,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,test
8989,1,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,test
8990,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,1,0,0,0,test


In [63]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('int64')], dtype=object),
 label
 0    0.917593
 1    0.082407
 Name: proportion, dtype: float64,
 subset
 train    0.699956
 test     0.300044
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.917717
         1        0.082283
 train   0        0.917541
         1        0.082459
 Name: proportion, dtype: float64,
 np.int64(0))

In [64]:
data.to_parquet("../Benchmark/qsar_oral_toxicity.parquet", index=False)

# QSAR androgen receptor


In [65]:
data = pd.read_csv("../Datasets/QSAR_androgen_receptor/qsar_androgen_receptor.csv", sep=";", header=None)
data

,0,1,2,3,4,5,6,7,8,9,...,1015,1016,1017,1018,1019,1020,1021,1022,1023,1024
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,positive
1,0,0,0,0,0,0,1,0,0,0,...,0,0,1,0,0,0,1,0,0,positive
2,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,positive
3,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,positive
4,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,positive
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1682,0,0,1,0,0,0,0,1,0,1,...,0,0,1,0,0,0,0,1,0,negative
1683,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,negative
1684,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,negative
1685,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,negative


In [66]:
data.rename(columns={1024: "label"}, inplace=True)

In [67]:
data.loc[data["label"] == "negative", "label"] = 0
data.loc[data["label"] == "positive", "label"] = 1
data["label"] = data["label"].astype(int)
data["label"].value_counts(normalize=True)

label
0    0.882039
1    0.117961
Name: proportion, dtype: float64

In [68]:
data

,0,1,2,3,4,5,6,7,8,9,...,1015,1016,1017,1018,1019,1020,1021,1022,1023,label
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,0,0,0,0,0,0,1,0,0,0,...,0,0,1,0,0,0,1,0,0,1
2,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1682,0,0,1,0,0,0,0,1,0,1,...,0,0,1,0,0,0,0,1,0,0
1683,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1684,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
1685,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["label"])

In [70]:
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data

,0,1,2,3,4,5,6,7,8,9,...,1016,1017,1018,1019,1020,1021,1022,1023,label,subset
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,train
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,train
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,train
3,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,train
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1682,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,test
1683,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,test
1684,0,0,0,0,0,0,0,0,0,0,...,1,0,1,0,0,0,0,1,0,test
1685,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,test


In [71]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('int64')], dtype=object),
 label
 0    0.882039
 1    0.117961
 Name: proportion, dtype: float64,
 subset
 train    0.699467
 test     0.300533
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.881657
         1        0.118343
 train   0        0.882203
         1        0.117797
 Name: proportion, dtype: float64,
 np.int64(0))

In [72]:
data.to_parquet("../Benchmark/qsar_androgen_receptor.parquet", index=False)

# Period Changer

In [73]:
data = pd.read_csv("../Datasets/Period_Changer/data.csv")
data

,MATS3v,nHBint10,MATS3s,MATS3p,nHBDon_Lipinski,minHBint8,MATS3e,MATS3c,minHBint2,MATS3m,...,WTPT-4,WTPT-5,ETA_EtaP_L,ETA_EtaP_F,ETA_EtaP_B,nT5Ring,SHdNH,ETA_dEpsilon_C,MDEO-22,Class
0,0.0908,0,0.0075,0.0173,0,0.0,-0.0436,0.0409,0.0,0.1368,...,0.0000,0.0000,0.1780,1.5488,0.0088,0,0.0,-0.0868,0.00,NoChanger
1,0.0213,0,0.1144,-0.0410,0,0.0,0.1231,-0.0316,0.0,0.1318,...,8.8660,19.3525,0.1739,1.3718,0.0048,2,0.0,-0.0810,0.25,NoChanger
2,0.0018,0,-0.0156,-0.0765,2,0.0,-0.1138,-0.1791,0.0,0.0615,...,5.2267,27.8796,0.1688,1.4395,0.0116,2,0.0,-0.1004,0.00,NoChanger
3,-0.0251,0,-0.0064,-0.0894,3,0.0,-0.0747,-0.1151,0.0,0.0361,...,7.7896,24.7336,0.1702,1.4654,0.0133,2,0.0,-0.1010,0.00,NoChanger
4,-0.0094,0,0.0132,-0.1035,0,0.0,-0.0046,-0.0870,0.0,0.1063,...,13.0472,7.0536,0.1785,1.4507,0.0113,2,0.0,-0.0824,0.00,NoChanger
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,-0.0960,0,-0.1131,-0.1086,1,0.0,-0.1250,0.0780,0.0,-0.2306,...,7.3247,6.9480,0.2213,1.0273,0.0172,2,0.0,-0.0812,0.00,Changer
86,-0.0944,0,-0.1736,-0.0182,0,0.0,-0.1476,0.0740,0.0,-0.2555,...,10.3324,6.9480,0.2003,1.1096,0.0172,2,0.0,-0.0960,0.00,Changer
87,-0.0868,0,-0.1118,-0.0858,1,0.0,-0.1440,0.0856,0.0,-0.1059,...,7.3260,6.9515,0.2277,1.0040,0.0203,2,0.0,-0.0886,0.00,Changer
88,-0.0146,0,-0.0783,-0.0251,1,0.0,-0.0518,0.0775,0.0,-0.1078,...,7.3308,6.9642,0.2430,1.0660,0.0154,2,0.0,-0.0570,0.00,Changer


In [74]:
data.rename(columns={"Class": "label"}, inplace=True)

In [75]:
data.loc[data["label"] == "NoChanger", "label"] = 0
data.loc[data["label"] == "Changer", "label"] = 1
data["label"] = data["label"].astype(int)
data["label"].value_counts(normalize=True)

label
0    0.7
1    0.3
Name: proportion, dtype: float64

In [76]:
data.columns = [list(range(len(data.columns))[:-1]) + ["label"]]
data = data[list(range(len(data.columns))[:-1]) + ["label"]].copy()
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data

,0,1,2,3,4,5,6,7,8,9,...,1168,1169,1170,1171,1172,1173,1174,1175,1176,label
0,0.0908,0,0.0075,0.0173,0,0.0,-0.0436,0.0409,0.0,0.1368,...,0.0000,0.0000,0.1780,1.5488,0.0088,0,0.0,-0.0868,0.00,0
1,0.0213,0,0.1144,-0.0410,0,0.0,0.1231,-0.0316,0.0,0.1318,...,8.8660,19.3525,0.1739,1.3718,0.0048,2,0.0,-0.0810,0.25,0
2,0.0018,0,-0.0156,-0.0765,2,0.0,-0.1138,-0.1791,0.0,0.0615,...,5.2267,27.8796,0.1688,1.4395,0.0116,2,0.0,-0.1004,0.00,0
3,-0.0251,0,-0.0064,-0.0894,3,0.0,-0.0747,-0.1151,0.0,0.0361,...,7.7896,24.7336,0.1702,1.4654,0.0133,2,0.0,-0.1010,0.00,0
4,-0.0094,0,0.0132,-0.1035,0,0.0,-0.0046,-0.0870,0.0,0.1063,...,13.0472,7.0536,0.1785,1.4507,0.0113,2,0.0,-0.0824,0.00,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,-0.0960,0,-0.1131,-0.1086,1,0.0,-0.1250,0.0780,0.0,-0.2306,...,7.3247,6.9480,0.2213,1.0273,0.0172,2,0.0,-0.0812,0.00,1
86,-0.0944,0,-0.1736,-0.0182,0,0.0,-0.1476,0.0740,0.0,-0.2555,...,10.3324,6.9480,0.2003,1.1096,0.0172,2,0.0,-0.0960,0.00,1
87,-0.0868,0,-0.1118,-0.0858,1,0.0,-0.1440,0.0856,0.0,-0.1059,...,7.3260,6.9515,0.2277,1.0040,0.0203,2,0.0,-0.0886,0.00,1
88,-0.0146,0,-0.0783,-0.0251,1,0.0,-0.0518,0.0775,0.0,-0.1078,...,7.3308,6.9642,0.2430,1.0660,0.0154,2,0.0,-0.0570,0.00,1


In [ ]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["label"])

In [78]:
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data

,0,1,2,3,4,5,6,7,8,9,...,1169,1170,1171,1172,1173,1174,1175,1176,label,subset
0,-0.0402,0,-0.0516,-0.0589,0,0.0000,-0.0555,-0.1234,0.0000,-0.0528,...,7.2107,0.2096,1.2534,0.0105,2,0.0,-0.0772,0.0,1,train
1,-0.0478,0,-0.1480,0.0147,1,0.0000,-0.1315,0.0831,0.0000,-0.2035,...,6.9480,0.2141,1.0552,0.0172,2,0.0,-0.0887,0.0,1,train
2,-0.0464,0,-0.0547,-0.0614,1,0.0000,-0.0504,-0.1835,5.3850,-0.0338,...,6.6292,0.2065,1.2646,0.0079,1,0.0,-0.0567,0.0,0,train
3,-0.0133,0,0.0146,-0.0652,0,0.0000,0.0460,-0.1150,0.0000,0.0639,...,22.9974,0.1805,1.3084,0.0040,2,0.0,-0.0639,0.0,0,train
4,-0.0028,2,-0.0164,-0.0912,2,0.0000,-0.0356,-0.0159,6.0139,0.0922,...,6.2113,0.1755,1.4774,0.0133,0,0.0,-0.0980,0.0,1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,-0.0371,0,0.0033,-0.1383,0,0.0000,-0.0289,-0.1308,0.0000,0.0831,...,6.6998,0.1782,1.2501,0.0099,1,0.0,-0.0963,0.0,0,test
86,-0.0002,0,0.0195,-0.0250,0,0.0000,-0.0746,0.0138,0.0000,0.0110,...,13.6686,0.1914,1.3260,0.0065,2,0.0,-0.0679,0.0,0,test
87,0.0934,0,0.1826,0.0578,0,0.0000,0.2267,0.1455,0.0000,0.1663,...,21.0268,0.2334,1.0597,0.0014,4,0.0,-0.0422,0.0,0,test
88,0.0444,0,-0.0185,-0.0413,2,5.9882,-0.0566,-0.2483,0.0000,0.0975,...,15.6943,0.1736,1.4299,0.0125,0,0.0,-0.1160,0.0,0,test


In [79]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('float64'), dtype('int64')], dtype=object),
 label
 0    0.7
 1    0.3
 Name: proportion, dtype: float64,
 subset
 train    0.7
 test     0.3
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.703704
         1        0.296296
 train   0        0.698413
         1        0.301587
 Name: proportion, dtype: float64,
 np.int64(0))

In [80]:
data.to_parquet("../Benchmark/period_changer.parquet", index=False)

# Toxicity

In [81]:
data = pd.read_csv("../Datasets/Toxicity/data.csv")
data

,MATS3v,nHBint10,MATS3s,MATS3p,nHBDon_Lipinski,minHBint8,MATS3e,MATS3c,minHBint2,MATS3m,...,WTPT-4,WTPT-5,ETA_EtaP_L,ETA_EtaP_F,ETA_EtaP_B,nT5Ring,SHdNH,ETA_dEpsilon_C,MDEO-22,Class
0,0.0908,0,0.0075,0.0173,0,0.0000,-0.0436,0.0409,0.0000,0.1368,...,0.0000,0.0000,0.1780,1.5488,0.0088,0,0.0,-0.0868,0.00,NonToxic
1,0.0213,0,0.1144,-0.0410,0,0.0000,0.1231,-0.0316,0.0000,0.1318,...,8.8660,19.3525,0.1739,1.3718,0.0048,2,0.0,-0.0810,0.25,NonToxic
2,0.0018,0,-0.0156,-0.0765,2,0.0000,-0.1138,-0.1791,0.0000,0.0615,...,5.2267,27.8796,0.1688,1.4395,0.0116,2,0.0,-0.1004,0.00,NonToxic
3,-0.0251,0,-0.0064,-0.0894,3,0.0000,-0.0747,-0.1151,0.0000,0.0361,...,7.7896,24.7336,0.1702,1.4654,0.0133,2,0.0,-0.1010,0.00,NonToxic
4,0.0135,0,0.0424,-0.0353,0,0.0000,-0.0638,0.0307,0.0000,0.0306,...,12.3240,19.7486,0.1789,1.4495,0.0120,2,0.0,-0.1071,0.00,NonToxic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166,-0.0960,0,-0.0478,-0.0840,2,0.0000,-0.0739,-0.2315,1.5660,-0.1133,...,2.5690,12.0174,0.1648,0.9710,0.0049,1,0.0,-0.0952,0.00,NonToxic
167,-0.0064,1,-0.1222,0.0013,1,0.0000,-0.1873,-0.2181,5.5404,-0.0757,...,10.7860,6.4871,0.1805,1.2298,0.0127,1,0.0,-0.0860,0.00,NonToxic
168,0.0096,2,-0.1846,0.0058,1,0.0000,-0.1293,-0.0979,5.3976,0.0409,...,4.9930,19.2864,0.2089,1.1245,0.0093,1,0.0,-0.0927,0.00,NonToxic
169,-0.0736,2,-0.1267,-0.0345,2,0.5346,-0.0361,0.0151,5.5190,-0.1025,...,10.7504,19.4989,0.1944,1.2256,0.0167,1,0.0,-0.1129,0.00,Toxic


In [82]:
data.rename(columns={"Class": "label"}, inplace=True)
data.loc[data["label"] == "NonToxic", "label"] = 0
data.loc[data["label"] == "Toxic", "label"] = 1
data["label"] = data["label"].astype(int)
data["label"].value_counts(normalize=True)

label
0    0.672515
1    0.327485
Name: proportion, dtype: float64

In [83]:
data.columns = [list(range(len(data.columns))[:-1]) + ["label"]]
data = data[list(range(len(data.columns))[:-1]) + ["label"]].copy()
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data

,0,1,2,3,4,5,6,7,8,9,...,1194,1195,1196,1197,1198,1199,1200,1201,1202,label
0,0.0908,0,0.0075,0.0173,0,0.0000,-0.0436,0.0409,0.0000,0.1368,...,0.0000,0.0000,0.1780,1.5488,0.0088,0,0.0,-0.0868,0.00,0
1,0.0213,0,0.1144,-0.0410,0,0.0000,0.1231,-0.0316,0.0000,0.1318,...,8.8660,19.3525,0.1739,1.3718,0.0048,2,0.0,-0.0810,0.25,0
2,0.0018,0,-0.0156,-0.0765,2,0.0000,-0.1138,-0.1791,0.0000,0.0615,...,5.2267,27.8796,0.1688,1.4395,0.0116,2,0.0,-0.1004,0.00,0
3,-0.0251,0,-0.0064,-0.0894,3,0.0000,-0.0747,-0.1151,0.0000,0.0361,...,7.7896,24.7336,0.1702,1.4654,0.0133,2,0.0,-0.1010,0.00,0
4,0.0135,0,0.0424,-0.0353,0,0.0000,-0.0638,0.0307,0.0000,0.0306,...,12.3240,19.7486,0.1789,1.4495,0.0120,2,0.0,-0.1071,0.00,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166,-0.0960,0,-0.0478,-0.0840,2,0.0000,-0.0739,-0.2315,1.5660,-0.1133,...,2.5690,12.0174,0.1648,0.9710,0.0049,1,0.0,-0.0952,0.00,0
167,-0.0064,1,-0.1222,0.0013,1,0.0000,-0.1873,-0.2181,5.5404,-0.0757,...,10.7860,6.4871,0.1805,1.2298,0.0127,1,0.0,-0.0860,0.00,0
168,0.0096,2,-0.1846,0.0058,1,0.0000,-0.1293,-0.0979,5.3976,0.0409,...,4.9930,19.2864,0.2089,1.1245,0.0093,1,0.0,-0.0927,0.00,0
169,-0.0736,2,-0.1267,-0.0345,2,0.5346,-0.0361,0.0151,5.5190,-0.1025,...,10.7504,19.4989,0.1944,1.2256,0.0167,1,0.0,-0.1129,0.00,1


In [ ]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["label"])

In [85]:
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data

,0,1,2,3,4,5,6,7,8,9,...,1195,1196,1197,1198,1199,1200,1201,1202,label,subset
0,-0.0419,0,-0.0343,-0.0501,2,5.9137,-0.0372,-0.0416,0.3860,-0.0433,...,16.5264,0.2317,1.1146,0.0153,2,0.0,-0.1018,0.0,0,train
1,-0.0846,2,-0.1705,-0.0603,2,0.5115,-0.1886,-0.0885,5.5939,-0.1550,...,22.4609,0.1830,1.2508,0.0152,2,0.0,-0.1530,0.0,0,train
2,-0.0064,1,-0.1222,0.0013,1,0.0000,-0.1873,-0.2181,5.5404,-0.0757,...,6.4871,0.1805,1.2298,0.0127,1,0.0,-0.0860,0.0,0,train
3,-0.0562,2,-0.0346,-0.0636,2,4.0160,-0.0739,-0.0369,0.0000,-0.0583,...,13.2295,0.2310,1.1427,0.0118,3,0.0,-0.0696,0.0,0,train
4,-0.0831,2,-0.0665,-0.0919,2,3.9352,-0.0668,-0.0075,0.0000,-0.0719,...,13.2233,0.2392,1.0963,0.0137,3,0.0,-0.0496,0.0,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166,-0.0528,0,0.0758,-0.0578,1,0.0000,0.0806,0.1523,0.0000,-0.0225,...,19.8346,0.2281,1.0578,0.0136,1,0.0,-0.0769,0.0,0,test
167,-0.0177,0,0.0096,-0.0994,1,0.0000,-0.0310,-0.0295,7.2910,0.0794,...,9.9445,0.1741,1.3236,0.0113,1,0.0,-0.0894,0.0,0,test
168,-0.0038,0,0.0170,-0.0905,2,7.7408,0.1151,-0.0399,7.7408,0.1552,...,26.5962,0.1784,1.4798,0.0098,0,0.0,-0.1137,0.0,1,test
169,-0.0552,0,0.0273,-0.0975,1,2.1399,-0.0088,0.0210,5.0960,0.0086,...,3.1149,0.1994,1.1296,0.0076,1,0.0,-0.0578,0.0,1,test


In [86]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('float64'), dtype('int64')], dtype=object),
 label
 0    0.672515
 1    0.327485
 Name: proportion, dtype: float64,
 subset
 train    0.695906
 test     0.304094
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.673077
         1        0.326923
 train   0        0.672269
         1        0.327731
 Name: proportion, dtype: float64,
 np.int64(0))

In [87]:
data.to_parquet("../Benchmark/toxicity.parquet", index=False)

# Swarm Behaviour

In [88]:
data = pd.read_csv("../Datasets/Swarm_Behaviour/Swarm Behavior Data/Aligned.csv")
data

,x1,y1,xVel1,yVel1,xA1,yA1,xS1,yS1,xC1,yC1,...,yVel200,xA200,yA200,xS200,yS200,xC200,yC200,nAC200,nS200,Class
0,-1414.14,-535.22,-17.88,-7.23,0.00,0.00,0.00,0.00,0.00,0.00,...,-16.85,0.0,0.00,0.00,0.00,0.00,0.00,29,0,0
1,-1412.93,597.54,-13.55,-5.48,0.00,0.00,0.00,0.00,0.00,0.00,...,-12.09,0.0,0.00,0.00,0.00,0.00,0.00,44,0,0
2,-1407.38,70.72,-14.37,-5.81,0.00,0.00,0.00,0.00,0.00,0.00,...,-16.20,0.0,0.00,0.00,0.00,0.00,0.00,40,0,0
3,-1407.00,-759.80,-7.59,-1.27,-0.98,-0.20,0.00,0.00,0.91,0.41,...,2.99,-1.0,-0.07,0.00,0.00,-0.52,0.86,3,0,1
4,-1406.36,698.39,-16.54,-6.95,-1.00,0.00,-944.07,-396.62,0.00,0.00,...,-12.61,0.0,-1.00,0.00,0.00,0.00,0.00,13,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24011,1403.71,948.55,4.54,-6.29,0.00,0.00,0.00,0.00,-0.13,-0.29,...,-4.87,0.0,0.00,0.00,0.00,-0.26,-0.18,11,0,0
24012,1403.72,133.09,9.46,14.33,0.00,1.00,0.00,0.00,0.00,0.00,...,5.20,0.0,1.00,-0.10,-3.24,0.00,0.00,29,0,0
24013,1404.38,144.31,6.98,3.89,0.00,0.00,0.00,0.00,0.00,0.00,...,9.50,0.0,0.00,-0.85,-0.52,0.00,0.00,5,1,0
24014,1404.61,-315.55,6.50,4.27,0.00,0.00,0.00,0.00,0.00,0.00,...,-0.75,0.0,0.00,0.00,0.00,0.00,0.00,1,0,0


In [89]:
data.rename(columns={"Class": "label"}, inplace=True)
data["label"].value_counts(normalize=True)

label
0    0.6875
1    0.3125
Name: proportion, dtype: float64

In [90]:
data.columns = [list(range(len(data.columns))[:-1]) + ["label"]]
data = data[list(range(len(data.columns))[:-1]) + ["label"]].copy()
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data

,0,1,2,3,4,5,6,7,8,9,...,2391,2392,2393,2394,2395,2396,2397,2398,2399,label
0,-1414.14,-535.22,-17.88,-7.23,0.00,0.00,0.00,0.00,0.00,0.00,...,-16.85,0.0,0.00,0.00,0.00,0.00,0.00,29,0,0
1,-1412.93,597.54,-13.55,-5.48,0.00,0.00,0.00,0.00,0.00,0.00,...,-12.09,0.0,0.00,0.00,0.00,0.00,0.00,44,0,0
2,-1407.38,70.72,-14.37,-5.81,0.00,0.00,0.00,0.00,0.00,0.00,...,-16.20,0.0,0.00,0.00,0.00,0.00,0.00,40,0,0
3,-1407.00,-759.80,-7.59,-1.27,-0.98,-0.20,0.00,0.00,0.91,0.41,...,2.99,-1.0,-0.07,0.00,0.00,-0.52,0.86,3,0,1
4,-1406.36,698.39,-16.54,-6.95,-1.00,0.00,-944.07,-396.62,0.00,0.00,...,-12.61,0.0,-1.00,0.00,0.00,0.00,0.00,13,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24011,1403.71,948.55,4.54,-6.29,0.00,0.00,0.00,0.00,-0.13,-0.29,...,-4.87,0.0,0.00,0.00,0.00,-0.26,-0.18,11,0,0
24012,1403.72,133.09,9.46,14.33,0.00,1.00,0.00,0.00,0.00,0.00,...,5.20,0.0,1.00,-0.10,-3.24,0.00,0.00,29,0,0
24013,1404.38,144.31,6.98,3.89,0.00,0.00,0.00,0.00,0.00,0.00,...,9.50,0.0,0.00,-0.85,-0.52,0.00,0.00,5,1,0
24014,1404.61,-315.55,6.50,4.27,0.00,0.00,0.00,0.00,0.00,0.00,...,-0.75,0.0,0.00,0.00,0.00,0.00,0.00,1,0,0


In [ ]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["label"])

In [92]:
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data

,0,1,2,3,4,5,6,7,8,9,...,2392,2393,2394,2395,2396,2397,2398,2399,label,subset
0,447.26,-7.18,-11.47,5.01,-0.96,-0.28,-1.27,1.33,-2.66,0.30,...,-0.93,-0.37,-3.04,1.48,2.68,0.13,93,6,0,train
1,542.19,-8.65,-2.84,-1.15,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,24,1,0,train
2,342.34,197.18,-0.43,8.91,-0.11,1.03,0.00,0.00,0.00,0.00,...,-0.30,1.00,0.00,0.00,0.00,0.00,12,0,1,train
3,-1078.07,-103.23,0.12,11.14,0.00,1.00,0.00,0.00,0.00,0.00,...,0.00,1.00,0.00,0.00,0.00,0.00,43,0,0,train
4,-1.47,380.39,-4.97,-4.94,0.00,-1.00,0.00,0.00,0.00,0.00,...,-1.00,-1.00,0.00,0.00,0.00,0.00,11,0,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24011,-378.92,-264.39,7.39,-6.69,0.00,0.00,0.00,0.00,0.28,-0.15,...,0.00,0.00,0.00,0.00,-0.29,-0.13,6,0,0,test
24012,-313.83,-958.01,-11.62,-4.70,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,17,1,0,test
24013,-58.93,-345.09,-13.25,-11.19,-1.00,-1.00,0.00,0.00,0.00,0.00,...,-1.00,-1.00,0.00,0.00,0.00,0.00,2,1,0,test
24014,1294.54,660.53,0.09,9.78,0.08,1.00,0.00,0.00,0.21,0.98,...,0.00,0.00,0.00,0.00,0.00,0.00,0,0,1,test


In [93]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('float64'), dtype('int64')], dtype=object),
 label
 0    0.6875
 1    0.3125
 Name: proportion, dtype: float64,
 subset
 train    0.699992
 test     0.300008
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.687439
         1        0.312561
 train   0        0.687526
         1        0.312474
 Name: proportion, dtype: float64,
 np.int64(0))

In [94]:
data.to_parquet("../Benchmark/swarm_behaviour_aligned.parquet", index=False)

# Gisette

In [95]:
input_train_data = pd.read_csv("../Datasets/Gisette/GISETTE/gisette_train.data", sep=" ", header=None).dropna(axis=1, how="all")
output_train_data = pd.read_csv("../Datasets/Gisette/GISETTE/gisette_train.labels", sep=" ", header=None, names=["label"])
train_data = pd.concat([input_train_data, output_train_data], axis=1).reset_index(drop=True)
train_data.loc[train_data["label"] == -1, "label"] = 0
train_data["subset"] = "train"
train_data

,0,1,2,3,4,5,6,7,8,9,...,4992,4993,4994,4995,4996,4997,4998,4999,label,subset
0,550,0,495,0,0,0,0,976,0,0,...,0,991,991,0,0,0,0,983,1,train
1,0,0,0,0,0,0,0,976,0,0,...,0,991,0,0,991,0,0,0,0,train
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,train
3,0,0,742,0,0,0,0,684,0,956,...,0,0,0,0,674,0,0,838,1,train
4,0,0,0,0,0,0,0,608,0,979,...,0,828,0,0,0,0,0,0,1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,783,0,0,0,0,train
5996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,921,0,886,0,1,train
5997,0,0,0,0,0,758,0,0,0,522,...,0,0,0,0,980,0,0,0,0,train
5998,0,0,0,0,0,0,0,0,0,0,...,0,0,690,0,0,0,0,0,0,train


In [96]:
input_test_data = pd.read_csv("../Datasets/Gisette/GISETTE/gisette_valid.data", sep=" ", header=None).dropna(axis=1, how="all")
output_test_data = pd.read_csv("../Datasets/Gisette/GISETTE/gisette_valid.labels", sep=" ", header=None, names=["label"])
test_data = pd.concat([input_test_data, output_test_data], axis=1).reset_index(drop=True)
test_data.loc[test_data["label"] == -1, "label"] = 0
test_data["subset"] = "test"
test_data

,0,1,2,3,4,5,6,7,8,9,...,4992,4993,4994,4995,4996,4997,4998,4999,label,subset
0,688,0,0,0,0,0,0,952,0,870,...,0,0,0,0,494,0,769,0,1,test
1,778,758,0,0,0,0,0,708,0,991,...,0,770,0,0,0,0,0,0,1,test
2,469,0,816,0,0,0,0,0,0,0,...,0,467,0,0,0,417,0,0,0,test
3,0,0,0,0,0,571,991,983,0,983,...,0,0,0,0,0,0,0,0,1,test
4,0,0,0,0,0,0,0,949,0,991,...,0,0,0,0,0,0,976,0,1,test
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0,511,0,0,0,0,0,0,0,0,...,0,852,0,0,0,0,0,0,0,test
996,0,0,599,0,0,0,0,910,0,0,...,0,0,0,0,0,0,0,707,1,test
997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,742,1,test
998,0,0,0,0,0,0,0,968,0,0,...,0,773,0,0,0,0,0,0,0,test


In [97]:
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data

,0,1,2,3,4,5,6,7,8,9,...,4992,4993,4994,4995,4996,4997,4998,4999,label,subset
0,550,0,495,0,0,0,0,976,0,0,...,0,991,991,0,0,0,0,983,1,train
1,0,0,0,0,0,0,0,976,0,0,...,0,991,0,0,991,0,0,0,0,train
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,train
3,0,0,742,0,0,0,0,684,0,956,...,0,0,0,0,674,0,0,838,1,train
4,0,0,0,0,0,0,0,608,0,979,...,0,828,0,0,0,0,0,0,1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6995,0,511,0,0,0,0,0,0,0,0,...,0,852,0,0,0,0,0,0,0,test
6996,0,0,599,0,0,0,0,910,0,0,...,0,0,0,0,0,0,0,707,1,test
6997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,742,1,test
6998,0,0,0,0,0,0,0,968,0,0,...,0,773,0,0,0,0,0,0,0,test


In [98]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('int64')], dtype=object),
 label
 1    0.5
 0    0.5
 Name: proportion, dtype: float64,
 subset
 train    0.857143
 test     0.142857
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.5
         1        0.5
 train   0        0.5
         1        0.5
 Name: proportion, dtype: float64,
 np.int64(0))

In [99]:
data.to_parquet("../Benchmark/gisette.parquet", index=False)

# REJAFADA

In [100]:
data = pd.read_csv("../Datasets/REJAFADA/REJAFADA.data", header=None)
data

,0,1,2,3,4,5,6,7,8,9,...,6816,6817,6818,6819,6820,6821,6822,6823,6824,6825
0,jar/benigns/100_2.json,B,1,1,1,2,5,18,18,55,...,0,0,0,1,0,1,1,0,0,0
1,jar/benigns/100_5.json,B,1,1,1,2,5,18,18,55,...,0,0,0,0,1,1,1,0,0,0
2,jar/benigns/100_6.json,B,1,1,1,2,5,18,18,55,...,0,0,0,0,1,1,1,0,0,0
3,jar/benigns/100_7.json,B,1,1,1,2,5,18,18,55,...,0,0,0,0,1,1,1,0,0,0
4,jar/benigns/100_8.json,B,1,1,1,2,5,18,18,55,...,0,0,0,0,1,1,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1991,jar/malwares/99_4.json,M,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
1992,jar/malwares/9_1.json,M,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
1993,jar/malwares/9_2.json,M,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
1994,jar/malwares/9_3.json,M,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0


In [101]:
data[0].nunique()

1996

In [102]:
data.rename(columns={1: "label"}, inplace=True)
data.drop(columns=[0], inplace=True)
data.loc[data["label"] == "B", "label"] = 0
data.loc[data["label"] == "M", "label"] = 1
data["label"] = data["label"].astype(int)
data["label"].value_counts(normalize=True)

label
0    0.5
1    0.5
Name: proportion, dtype: float64

In [103]:
data

,label,2,3,4,5,6,7,8,9,10,...,6816,6817,6818,6819,6820,6821,6822,6823,6824,6825
0,0,1,1,1,2,5,18,18,55,48,...,0,0,0,1,0,1,1,0,0,0
1,0,1,1,1,2,5,18,18,55,48,...,0,0,0,0,1,1,1,0,0,0
2,0,1,1,1,2,5,18,18,55,48,...,0,0,0,0,1,1,1,0,0,0
3,0,1,1,1,2,5,18,18,55,48,...,0,0,0,0,1,1,1,0,0,0
4,0,1,1,1,2,5,18,18,55,48,...,0,0,0,0,1,1,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1991,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
1992,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
1993,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
1994,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0


In [104]:
data.columns = [["label"] + list(range(len(data.columns))[:-1])]
data = data[list(range(len(data.columns))[:-1]) + ["label"]].copy()
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data

,0,1,2,3,4,5,6,7,8,9,...,6815,6816,6817,6818,6819,6820,6821,6822,6823,label
0,1,1,1,2,5,18,18,55,48,1,...,0,0,1,0,1,1,0,0,0,0
1,1,1,1,2,5,18,18,55,48,1,...,0,0,0,1,1,1,0,0,0,0
2,1,1,1,2,5,18,18,55,48,1,...,0,0,0,1,1,1,0,0,0,0
3,1,1,1,2,5,18,18,55,48,1,...,0,0,0,1,1,1,0,0,0,0
4,1,1,1,2,5,18,18,55,48,1,...,0,0,0,1,1,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1991,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
1992,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
1993,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
1994,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1


In [ ]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["label"])

In [106]:
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data

,0,1,2,3,4,5,6,7,8,9,...,6816,6817,6818,6819,6820,6821,6822,6823,label,subset
0,0,0,0,0,0,0,0,0,0,0,...,0,0,1,1,1,0,0,0,0,train
1,0,0,0,0,0,0,0,0,0,0,...,0,0,1,1,1,0,0,0,0,train
2,0,0,0,0,0,0,0,0,0,0,...,0,0,1,1,1,0,0,0,1,train
3,0,0,0,0,0,0,0,0,0,0,...,0,0,1,1,1,0,0,0,0,train
4,0,0,0,0,0,0,0,0,0,0,...,0,0,1,1,1,0,0,0,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1991,0,0,0,0,0,0,0,0,0,0,...,0,0,1,1,1,0,0,0,0,test
1992,0,0,0,0,0,0,0,0,0,0,...,0,0,1,1,1,0,0,0,0,test
1993,0,0,0,0,0,0,0,0,0,0,...,0,0,1,1,1,0,0,0,0,test
1994,0,0,0,0,0,0,0,0,0,0,...,0,0,1,1,1,0,0,0,0,test


In [107]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('int64')], dtype=object),
 label
 0    0.5
 1    0.5
 Name: proportion, dtype: float64,
 subset
 train    0.6999
 test     0.3001
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.500835
         1        0.499165
 train   1        0.500358
         0        0.499642
 Name: proportion, dtype: float64,
 np.int64(0))

In [108]:
data.to_parquet("../Benchmark/rejafada.parquet", index=False)

# Arcene

In [109]:
input_train_data = pd.read_csv("../Datasets/Arcene/ARCENE/arcene_train.data", sep=" ", header=None).dropna(axis=1, how="all")
output_train_data = pd.read_csv("../Datasets/Arcene/ARCENE/arcene_train.labels", sep=" ", header=None, names=["label"])
train_data = pd.concat([input_train_data, output_train_data], axis=1).reset_index(drop=True)
train_data.loc[train_data["label"] == -1, "label"] = 0
train_data["subset"] = "train"
train_data

,0,1,2,3,4,5,6,7,8,9,...,9992,9993,9994,9995,9996,9997,9998,9999,label,subset
0,0,71,0,95,0,538,404,20,0,0,...,86,0,36,0,80,0,0,524,1,train
1,0,41,82,165,60,554,379,0,71,0,...,69,7,473,0,57,0,284,423,0,train
2,0,0,1,40,0,451,402,0,0,0,...,28,0,24,0,90,0,34,508,1,train
3,0,56,44,275,14,511,470,0,0,0,...,0,26,86,0,102,0,0,469,1,train
4,105,0,141,348,0,268,329,0,0,1,...,0,0,0,190,301,0,0,354,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,24,26,0,461,0,545,0,17,159,177,...,22,26,130,306,182,0,94,336,0,train
96,40,0,0,419,71,502,0,39,93,163,...,0,68,61,295,133,0,0,292,0,train
97,2,15,48,677,0,434,442,0,43,0,...,0,7,228,0,105,0,0,453,1,train
98,8,0,38,205,69,419,454,0,113,3,...,148,27,656,0,133,0,189,403,0,train


In [110]:
input_test_data = pd.read_csv("../Datasets/Arcene/ARCENE/arcene_valid.data", sep=" ", header=None).dropna(axis=1, how="all")
output_test_data = pd.read_csv("../Datasets/Arcene/ARCENE/arcene_valid.labels", sep=" ", header=None, names=["label"])
test_data = pd.concat([input_test_data, output_test_data], axis=1).reset_index(drop=True)
test_data.loc[test_data["label"] == -1, "label"] = 0
test_data["subset"] = "test"
test_data

,0,1,2,3,4,5,6,7,8,9,...,9992,9993,9994,9995,9996,9997,9998,9999,label,subset
0,0,0,156,138,2,635,444,0,1,0,...,42,0,50,67,87,0,0,465,0,test
1,0,7,0,7,0,251,0,0,22,222,...,0,0,18,201,317,0,34,199,0,test
2,0,32,0,470,53,493,0,0,140,147,...,20,34,165,225,53,0,47,219,0,test
3,0,77,0,202,0,432,472,0,60,0,...,13,0,488,0,71,0,177,487,1,test
4,0,34,102,522,2,474,473,0,39,0,...,0,29,192,0,74,0,5,416,1,test
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,24,73,0,436,92,400,0,0,139,261,...,0,86,130,365,58,17,3,37,0,test
96,11,58,50,332,109,393,122,0,75,134,...,156,77,26,277,265,0,36,261,0,test
97,93,32,137,319,0,264,231,21,0,0,...,0,0,0,244,309,0,276,312,1,test
98,119,12,198,339,0,289,410,0,0,4,...,37,0,0,256,402,0,0,350,1,test


In [111]:
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data

,0,1,2,3,4,5,6,7,8,9,...,9992,9993,9994,9995,9996,9997,9998,9999,label,subset
0,0,71,0,95,0,538,404,20,0,0,...,86,0,36,0,80,0,0,524,1,train
1,0,41,82,165,60,554,379,0,71,0,...,69,7,473,0,57,0,284,423,0,train
2,0,0,1,40,0,451,402,0,0,0,...,28,0,24,0,90,0,34,508,1,train
3,0,56,44,275,14,511,470,0,0,0,...,0,26,86,0,102,0,0,469,1,train
4,105,0,141,348,0,268,329,0,0,1,...,0,0,0,190,301,0,0,354,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,24,73,0,436,92,400,0,0,139,261,...,0,86,130,365,58,17,3,37,0,test
196,11,58,50,332,109,393,122,0,75,134,...,156,77,26,277,265,0,36,261,0,test
197,93,32,137,319,0,264,231,21,0,0,...,0,0,0,244,309,0,276,312,1,test
198,119,12,198,339,0,289,410,0,0,4,...,37,0,0,256,402,0,0,350,1,test


In [112]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('int64')], dtype=object),
 label
 0    0.56
 1    0.44
 Name: proportion, dtype: float64,
 subset
 train    0.5
 test     0.5
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.56
         1        0.44
 train   0        0.56
         1        0.44
 Name: proportion, dtype: float64,
 np.int64(0))

In [113]:
data.to_parquet("../Benchmark/arcene.parquet", index=False)

# Dorothea

In [114]:
def load_dorothea_dataset(base_path: str, n_features: int = 100000) -> pd.DataFrame:
    """Load Dorothea dataset (train/valid/test) into a pandas sparse DataFrame.
    
    Parameters
    ----------
    base_path : str
        Path to the dataset, without file extension (e.g., 'dorothea_train')
    n_features : int, optional
        Total number of features (default: 100000)

    Returns
    -------
    pd.DataFrame
        Sparse pandas DataFrame containing all features and the target column (if labels file exists)
    """
    data_path = f"{base_path}.data"
    labels_path = f"{base_path}.labels"

    # Read .data file
    rows = []
    with open(data_path) as f:
        for line in f:
            if not line.strip():
                continue
            indices = [int(x) - 1 for x in line.split()]  # 0-based indices
            rows.append(indices)

    # Build sparse matrix
    X = lil_matrix((len(rows), n_features), dtype=np.uint8)
    for i, indices in enumerate(rows):
        X[i, indices] = 1

    # Convert to pandas sparse DataFrame
    df = pd.DataFrame.sparse.from_spmatrix(X)

    # Add labels if available
    try:
        y = np.loadtxt(labels_path)
        df["label"] = y
    except FileNotFoundError:
        pass  # test set usually has no labels

    return df

In [115]:
train_data = load_dorothea_dataset("../Datasets/Dorothea drug discovery/dorothea_train")
train_data.loc[train_data["label"] == -1, "label"] = 0
train_data["label"] = train_data["label"].astype(int)
train_data = train_data.apply(lambda col: col.sparse.to_dense() if isinstance(col.dtype, pd.SparseDtype) else col)
train_data = train_data.astype(np.uint8)
train_data["subset"] = "train"

In [116]:
test_data = load_dorothea_dataset("../Datasets/Dorothea drug discovery/dorothea_valid")
test_data.loc[test_data["label"] == -1, "label"] = 0
test_data["label"] = test_data["label"].astype(int)
test_data = test_data.apply(lambda col: col.sparse.to_dense() if isinstance(col.dtype, pd.SparseDtype) else col)
test_data = test_data.astype(np.uint8)
test_data["subset"] = "test"

In [117]:
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data

,0,1,2,3,4,5,6,7,8,9,...,99992,99993,99994,99995,99996,99997,99998,99999,label,subset
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,train
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,train
2,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,train
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,train
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1145,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,test
1146,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,test
1147,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,test
1148,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,test


In [118]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('uint8')], dtype=object),
 label
 0    0.902609
 1    0.097391
 Name: proportion, dtype: float64,
 subset
 train    0.695652
 test     0.304348
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.902857
         1        0.097143
 train   0        0.902500
         1        0.097500
 Name: proportion, dtype: float64,
 np.int64(0))

In [119]:
data.to_parquet("../Benchmark/dorothea.parquet", index=False)

# Dexter

In [120]:
def load_dexter_dataset(path: str, n_features: int = 20000) -> pd.DataFrame:
    """Load the Dexter dataset from a file containing feature-only entries.

    Parameters
    ----------
    path : str
        Path to the `.data` file (e.g., `"../Datasets/Dexter/DEXTER/dexter_train.data"`).
    n_features : int, default=20000
        Total number of features in the dataset. Dexter officially contains 20,000 features.

    Returns
    -------
    pd.DataFrame
        A pandas SparseDataFrame where each column corresponds to a feature and each row
        represents a sample. The `"target"` column contains the class labels (+1 or -1) if
        the `.labels` file is found.
    """
    data, row_ind, col_ind = [], [], []

    with open(path + ".data", "r") as f:
        for row_idx, line in enumerate(f):
            parts = line.strip().split()
            for p in parts:
                idx, val = p.split(":")
                col_ind.append(int(idx) - 1)
                data.append(float(val))
                row_ind.append(row_idx)

    X = coo_matrix((data, (row_ind, col_ind)), shape=(row_idx + 1, n_features), dtype=np.float32)
    df = pd.DataFrame.sparse.from_spmatrix(X)

    # --- Try to load labels ---
    labels_path = path + ".labels"
    y = np.loadtxt(labels_path)
    df["label"] = y

    return df

In [121]:
train_data = load_dexter_dataset("../Datasets/Dexter/DEXTER/dexter_train")
train_data.loc[train_data["label"] == -1, "label"] = 0
train_data["label"] = train_data["label"].astype(int)
train_data = train_data.apply(lambda col: col.sparse.to_dense() if isinstance(col.dtype, pd.SparseDtype) else col)
train_data = train_data.astype(np.uint8)
train_data["subset"] = "train"

In [122]:
train_data

,0,1,2,3,4,5,6,7,8,9,...,19992,19993,19994,19995,19996,19997,19998,19999,label,subset
0,0,0,0,0,0,0,0,0,0,105,...,0,0,0,0,0,0,56,0,1,train
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,train
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,train
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,train
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,train
296,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,train
297,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,train
298,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,train


In [123]:
test_data = load_dexter_dataset("../Datasets/Dexter/DEXTER/dexter_valid")
test_data.loc[test_data["label"] == -1, "label"] = 0
test_data["label"] = test_data["label"].astype(int)
train_dtest_dataata = test_data.apply(lambda col: col.sparse.to_dense() if isinstance(col.dtype, pd.SparseDtype) else col)
test_data = test_data.astype(np.uint8)
test_data["subset"] = "test"

C:\Users\pedbr\AppData\Local\Temp\ipykernel_52732\398417537.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_data["subset"] = "test"


In [124]:
test_data

,0,1,2,3,4,5,6,7,8,9,...,19992,19993,19994,19995,19996,19997,19998,19999,label,subset
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,test
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,test
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,test
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,test
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,test
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,test
296,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,test
297,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,test
298,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,test


In [125]:
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data

,0,1,2,3,4,5,6,7,8,9,...,19992,19993,19994,19995,19996,19997,19998,19999,label,subset
0,0,0,0,0,0,0,0,0,0,105,...,0,0,0,0,0,0,56,0,1,train
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,train
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,train
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,train
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,test
596,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,test
597,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,test
598,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,test


In [126]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('uint8')], dtype=object),
 label
 1    0.5
 0    0.5
 Name: proportion, dtype: float64,
 subset
 train    0.5
 test     0.5
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.5
         1        0.5
 train   0        0.5
         1        0.5
 Name: proportion, dtype: float64,
 np.int64(0))

In [127]:
data.to_parquet("../Benchmark/dexter.parquet", index=False)